# Costos y Beneficios asociados a la implementación del SMI por Comuna (Resultados)

Este notebook refleja los resultados documentados en 'CB_implementacion_SMI_Chile.ipynb' pero sin argumentar los costos ni detallar la forma del cálculo para efectos prácticos de la obtención de resultados.

In [53]:
import pandas as pd 
import numpy as np
import unidecode as ud
from pprint import pprint


df_caracterizacion_distribucion = pd.read_excel("./Para obtener la caracterización de Dx/Salidas/Caracterizacion_Dx_Dic2024_comuna (CNE).xlsx")

# Dolar y UF considerados (a 16 de Oct de 2025, según SII)
usd = 959.36
uf 	= 39_521.26

# Se elimina la columna "Cantidad de alimentadores" que no es necesaria para el análisis (no incluidas en el SMMC de la Norma)
df_caracterizacion_distribucion = df_caracterizacion_distribucion.drop(columns=["Cantidad de alimentadores",
																				"Energía facturada [USD]"])

# Se setea la comuna y el segmento como índice del DataFrame
df_caracterizacion_distribucion = df_caracterizacion_distribucion.set_index(["Comuna", "Segmento"], drop=False)

# Se anualiza la Energía facturada
df_caracterizacion_distribucion["Energía promedio mensual [kWh]"] = (df_caracterizacion_distribucion["Energía promedio mensual [kWh]"] * 12) / 1000  # Conversión a MWh/año

# Se renombran columnas para mayor claridad
df_caracterizacion_distribucion = df_caracterizacion_distribucion.rename(columns={
	"Energía promedio mensual [kWh]": "Energía facturada anual [MWh/año]"
})

# Decisión de la tecnología a ocupar; PLC, Celular, RF, LoRa, etc.
df_implementacion = df_caracterizacion_distribucion.copy()

# Se definen los niveles de densidad que existen en la tabla
niveles_densidad = [
	"EXTREMADAMENTE BAJA",
	"MUY BAJA",
	"BAJA",
	"MEDIA",
	"ALTA"
]

print("Se asigna la tecnología de comunicaciones para cada nivel de densidad")
print("Tecnologías posibles: 'Celular', 'TWACS', 'G3-PLC', 'RF-Mesh', 'LoRa'")

# Input de tecnología y armando del mapeo
mapeo_tecnologia = {}

# mapeo_tecnologia["EXTREMADAMENTE BAJA"] = "Celular"
# mapeo_tecnologia["MUY BAJA"] = "Celular"
# mapeo_tecnologia["BAJA"] = "Celular"
# mapeo_tecnologia["MEDIA"] = "Celular"
# mapeo_tecnologia["ALTA"] = "Celular"

# mapeo_tecnologia["EXTREMADAMENTE BAJA"] = "TWACS"
# mapeo_tecnologia["MUY BAJA"] = "TWACS"
# mapeo_tecnologia["BAJA"] = "TWACS"
# mapeo_tecnologia["MEDIA"] = "TWACS"
# mapeo_tecnologia["ALTA"] = "TWACS"

# mapeo_tecnologia["EXTREMADAMENTE BAJA"] = "G3-PLC"
# mapeo_tecnologia["MUY BAJA"] = "G3-PLC"
# mapeo_tecnologia["BAJA"] = "G3-PLC"
# mapeo_tecnologia["MEDIA"] = "G3-PLC"
# mapeo_tecnologia["ALTA"] = "G3-PLC"

# mapeo_tecnologia["EXTREMADAMENTE BAJA"] = "RF-Mesh"
# mapeo_tecnologia["MUY BAJA"] = "RF-Mesh"
# mapeo_tecnologia["BAJA"] = "RF-Mesh"
# mapeo_tecnologia["MEDIA"] = "RF-Mesh"
# mapeo_tecnologia["ALTA"] = "RF-Mesh"

# mapeo_tecnologia["EXTREMADAMENTE BAJA"] = "LoRa"
# mapeo_tecnologia["MUY BAJA"] = "LoRa"
# mapeo_tecnologia["BAJA"] = "LoRa"
# mapeo_tecnologia["MEDIA"] = "LoRa"
# mapeo_tecnologia["ALTA"] = "LoRa"

mapeo_tecnologia["EXTREMADAMENTE BAJA"] = "LoRa"
mapeo_tecnologia["MUY BAJA"] = "LoRa"
mapeo_tecnologia["BAJA"] = "RF-Mesh"
mapeo_tecnologia["MEDIA"] = "G3-PLC"
mapeo_tecnologia["ALTA"] = "G3-PLC"

# Se crea la nueva columna usando la columna 'Densidad'
df_implementacion["Tecnología"] = df_implementacion["Densidad"].map(mapeo_tecnologia)

Se asigna la tecnología de comunicaciones para cada nivel de densidad
Tecnologías posibles: 'Celular', 'TWACS', 'G3-PLC', 'RF-Mesh', 'LoRa'


## Requerimientos sobre la cantidad de equipos necesarios por comuna y segmento

In [54]:
# Se crea el df asociado a una implementación real temporal
df_implementacion_1 = df_implementacion.copy()		# Año 1 (2026)
df_implementacion_2 = df_implementacion.copy()		# Año 2 (2027)
df_implementacion_3 = df_implementacion.copy()		# Año 3 (2028)
df_implementacion_4 = df_implementacion.copy()		# Año 4 (2029)
df_implementacion_5 = df_implementacion.copy()		# Año 5 (2030)
df_implementacion_6 = df_implementacion.copy()		# Año 6 (2031)
df_implementacion_7 = df_implementacion.copy()		# Año 7 (2032)
df_implementacion_8 = df_implementacion.copy()		# Año 8 (2033)
df_implementacion_9 = df_implementacion.copy()		# Año 9 (2034)
df_implementacion_10 = df_implementacion.copy()		# Año 10 (2035)
df_implementacion_11 = df_implementacion.copy()		# Año 11 (2036)
df_implementacion_12 = df_implementacion.copy()		# Año 12 (2037)
df_implementacion_13 = df_implementacion.copy()		# Año 13 (2038)
df_implementacion_14 = df_implementacion.copy()		# Año 14 (2039)
df_implementacion_15 = df_implementacion.copy()		# Año 15 (2040)

lista_dfs_implementacion = [df_implementacion,
							df_implementacion_1,
							df_implementacion_2,
							df_implementacion_3,
							df_implementacion_4,
							df_implementacion_5,
							df_implementacion_6,
							df_implementacion_7,
							df_implementacion_8,
							df_implementacion_9,
							df_implementacion_10,
							df_implementacion_11,
							df_implementacion_12,
							df_implementacion_13,
							df_implementacion_14,
							df_implementacion_15]


crecimiento_anual_demanda = 0.0302  			# Crecimiento anual de la demanda (3.02%)

## Definición de factores de crecimiento para cada segmento y densidad
crecimiento_anual_residencial = np.where(df_implementacion["Densidad"] == 'ALTA', 0.0137,
                                         np.where(df_implementacion["Densidad"] == 'MEDIA', 0.0181,
												  np.where(df_implementacion["Densidad"] == 'BAJA', 0.0084,
														   np.where(df_implementacion["Densidad"] == 'MUY BAJA', 0.0172, 
																	np.where(df_implementacion["Densidad"] == 'EXTREMADAMENTE BAJA', 0.0150, 0)))))

crecimiento_anual_no_residencial = np.where(df_implementacion["Densidad"] == 'ALTA', 0.0013,
                                         np.where(df_implementacion["Densidad"] == 'MEDIA', 0.0100,
												  np.where(df_implementacion["Densidad"] == 'BAJA', 0.0125,
														   np.where(df_implementacion["Densidad"] == 'MUY BAJA', 0.0141, 
																	np.where(df_implementacion["Densidad"] == 'EXTREMADAMENTE BAJA', 0.0014, 0)))))

crecimiento_anual_libredx = np.where(df_implementacion["Densidad"] == 'ALTA', 0.0201,
                                         np.where(df_implementacion["Densidad"] == 'MEDIA', 0.0388,
												  np.where(df_implementacion["Densidad"] == 'BAJA', 0.0326,
														   np.where(df_implementacion["Densidad"] == 'MUY BAJA', 0.0480, 
																	np.where(df_implementacion["Densidad"] == 'EXTREMADAMENTE BAJA', 0.0227, 0)))))

crecimiento_anual_TD = np.where(df_implementacion["Densidad"] == 'ALTA', 0.0035,
                                         np.where(df_implementacion["Densidad"] == 'MEDIA', 0.0050,
												  np.where(df_implementacion["Densidad"] == 'BAJA', 0.0011,
														   np.where(df_implementacion["Densidad"] == 'MUY BAJA', 0.0061, 
																	np.where(df_implementacion["Densidad"] == 'EXTREMADAMENTE BAJA', 0.0003, 0)))))


# Los datos utilizados de energía y cantidad de clientes corresponden al año 2024 (df_implementacion), sin embargo,
# el año base de la implementación es 2025 (df_implementacion_0). Salvo para el caso de lo clientes libres el cual si está actualizado a 2025.
factor_crecimiento_cantidad_clientes = np.where(df_implementacion["Segmento"] == 'Residencial', crecimiento_anual_residencial,
											np.where(df_implementacion["Segmento"] == 'No Residencial BT', crecimiento_anual_residencial,
												np.where(df_implementacion["Segmento"] == 'No Residencial AT', crecimiento_anual_no_residencial, 0)))

factor_crecimiento_cantidad_TD = np.where(df_implementacion["Segmento"] == 'Red', crecimiento_anual_TD, 0)

factor_crecimiento_demanda = np.where(df_implementacion["Segmento"] == 'Residencial', crecimiento_anual_demanda,
								np.where(df_implementacion["Segmento"] == 'No Residencial BT', crecimiento_anual_demanda,
									np.where(df_implementacion["Segmento"] == 'No Residencial AT', crecimiento_anual_demanda, 0)))

# Se actualizan los valores de "Cantidad de clientes" según el crecimiento anual esperado (factor compuesto)
lista_dfs_implementacion[0]["Cantidad de clientes"] = ((lista_dfs_implementacion[0]["Cantidad de clientes"]
												* (1 + factor_crecimiento_cantidad_clientes)).round().astype('Int64'))

lista_dfs_implementacion[0]["UM 1F para TD"] = ((lista_dfs_implementacion[0]["UM 1F para TD"]
											* (1 + factor_crecimiento_cantidad_TD)).round().astype('Int64'))

lista_dfs_implementacion[0]["UM 3F 1S para TD"] = ((lista_dfs_implementacion[0]["UM 3F 1S para TD"]
											* (1 + factor_crecimiento_cantidad_TD)).round().astype('Int64'))

lista_dfs_implementacion[0]["UM 3F 2S para TD"] = ((lista_dfs_implementacion[0]["UM 3F 2S para TD"]
											* (1 + factor_crecimiento_cantidad_TD)).round().astype('Int64'))

# Se actualiza la Energía facturada anual según el crecimiento anual esperado (factor compuesto)
lista_dfs_implementacion[0]["Energía facturada anual [MWh/año]"] = (lista_dfs_implementacion[0]["Energía facturada anual [MWh/año]"]
															* (1 + factor_crecimiento_demanda))


### Se calclula la cantidad de Unidades de Medida (UM) necesarias
for periodo in range(len(lista_dfs_implementacion)):

	# Se agrega la columna de "Cantidad UM 1F" según el segmento
	valores = []

	for i, row in lista_dfs_implementacion[periodo].iterrows():
		
		if row['Segmento'] == 'Residencial' or row['Segmento'] == 'No Residencial BT':
			valores.append(row['Cantidad de clientes'])
		elif row['Segmento'] == 'Red':
			valores.append(row['UM 1F para TD'])
		else:
			valores.append(0)

	lista_dfs_implementacion[periodo]["Cantidad UM 1F"] = pd.Series(valores, index=lista_dfs_implementacion[periodo].index).round().astype('Int64')

	# Se agrega la columna de "Cantidad UM 3F Directa" según el segmento
	valores = []

	for i, row in lista_dfs_implementacion[periodo].iterrows():
		
		if row['Segmento'] == 'No Residencial AT':
			valores.append(row['Cantidad de clientes'])
		else:
			valores.append(0)

	lista_dfs_implementacion[periodo]["Cantidad UM 3F Directa"] = pd.Series(valores, index=lista_dfs_implementacion[periodo].index).round().astype('Int64')

	# Se agrega la columna de "Cantidad UM 3F Indirecta" según el segmento
	valores = []

	for i, row in lista_dfs_implementacion[periodo].iterrows():
		
		if row['Segmento'] == 'Red' or row['Segmento'] == 'LibreDx':
			valores.append(row['UM 3F 1S para TD'] + row['UM 3F 2S para TD'] * 2 + row['Cantidad de clientes'])
		else:
			valores.append(0)

	lista_dfs_implementacion[periodo]["Cantidad UM 3F Indirecta"] = pd.Series(valores, index=lista_dfs_implementacion[periodo].index).round().astype('Int64')

	# Se preparan los datos para el siguiente período sin afectar el actual (factor compuesto)
	lista_dfs_implementacion[periodo]["Cantidad de clientes"] = (lista_dfs_implementacion[0]["Cantidad de clientes"]
															  	 * (1 + factor_crecimiento_cantidad_clientes) ** periodo).round().astype('Int64')
	
	lista_dfs_implementacion[periodo]["UM 1F para TD"] = (lista_dfs_implementacion[0]["UM 1F para TD"]
															  	 * (1 + factor_crecimiento_cantidad_TD) ** periodo).round().astype('Int64')
	
	lista_dfs_implementacion[periodo]["UM 3F 1S para TD"] = (lista_dfs_implementacion[0]["UM 3F 1S para TD"]
															  	 * (1 + factor_crecimiento_cantidad_TD) ** periodo).round().astype('Int64')
	
	lista_dfs_implementacion[periodo]["UM 3F 2S para TD"] = (lista_dfs_implementacion[0]["UM 3F 2S para TD"]
															  	 * (1 + factor_crecimiento_cantidad_TD) ** periodo).round().astype('Int64')

	if periodo + 1 < len(lista_dfs_implementacion):
		
		# Se actualizan los valores de "Cantidad de clientes" según el crecimiento anual esperado
		factor_crecimiento = np.where(lista_dfs_implementacion[periodo]["Segmento"] == 'Residencial', crecimiento_anual_residencial,
								np.where(lista_dfs_implementacion[periodo]["Segmento"] == 'No Residencial BT', crecimiento_anual_residencial,
				 				np.where(lista_dfs_implementacion[periodo]["Segmento"] == 'No Residencial AT', crecimiento_anual_no_residencial,
								np.where(lista_dfs_implementacion[periodo]["Segmento"] == 'LibreDx', crecimiento_anual_libredx,
				 				np.where(lista_dfs_implementacion[periodo]["Segmento"] == 'Red', crecimiento_anual_TD,
								0)))))
		
		# Se actualizan los valores de "Cantidad de clientes" y "Cantidad de TD" según el crecimiento anual esperado (factor compuesto)
		lista_dfs_implementacion[periodo + 1]["Cantidad de clientes"] = (((lista_dfs_implementacion[periodo]["Cantidad de clientes"]
																		* (1 + factor_crecimiento)).round().astype('Int64')) 
																		- (lista_dfs_implementacion[periodo]["Cantidad de clientes"]))
		
		lista_dfs_implementacion[periodo + 1]["UM 1F para TD"] = (((lista_dfs_implementacion[periodo]["UM 1F para TD"]
																		* (1 + factor_crecimiento)).round().astype('Int64')) 
																		- (lista_dfs_implementacion[periodo]["UM 1F para TD"]))
		
		lista_dfs_implementacion[periodo + 1]["UM 3F 1S para TD"] = (((lista_dfs_implementacion[periodo]["UM 3F 1S para TD"]
																		* (1 + factor_crecimiento)).round().astype('Int64')) 
																		- (lista_dfs_implementacion[periodo]["UM 3F 1S para TD"]))
		
		lista_dfs_implementacion[periodo + 1]["UM 3F 2S para TD"] = (((lista_dfs_implementacion[periodo]["UM 3F 2S para TD"]
																		* (1 + factor_crecimiento)).round().astype('Int64')) 
																		- (lista_dfs_implementacion[periodo]["UM 3F 2S para TD"]))

		# Se actualiza la Energía facturada anual según el crecimiento anual esperado (factor compuesto)
		lista_dfs_implementacion[periodo + 1]["Energía facturada anual [MWh/año]"] = (lista_dfs_implementacion[periodo]["Energía facturada anual [MWh/año]"]
																					* (1 + crecimiento_anual_demanda))
		

### Se calclula la cantidad de Unidades Concentradoras (DCU) necesarias
valores = []

for i, row in df_implementacion.iterrows():
	
	if row["Tecnología"] == "G3-PLC" and row["Segmento"] == "Red":
		dcu_requeridas = (row['UM 1F para TD'] + row['UM 3F 1S para TD'] + row['UM 3F 2S para TD'])		# Sujeto a cambios según estimación de TD
		valores.append(dcu_requeridas)
	
	elif row["Tecnología"] == "RF-Mesh" and row["Segmento"] == "Red":
		capacidad = 1000  																								# Capacidad de 1000 equipos por DCU
		cantidad_cliente_comuna = df_implementacion.loc[(row["Comuna"], slice(None)), "Cantidad de clientes"].sum()			
		dcu_requeridas = (cantidad_cliente_comuna // capacidad) + 1  													# Redondeo hacia arriba
		valores.append(dcu_requeridas)

	elif row["Tecnología"] == "LoRa" and row["Segmento"] == "Red":
		if row["Densidad"] == "EXTREMADAMENTE BAJA" or row["Densidad"] == "MUY BAJA" or row["Densidad"] == "BAJA":
			cobertura_km2 = (15 ** 2) * np.pi 
			dcu_requeridas = np.ceil(row["Superficie efectiva [km2]"] / cobertura_km2)					# Rural 15 km
			valores.append(dcu_requeridas) 	
		else:
			cobertura_km2 = (2 ** 2) * np.pi
			dcu_requeridas = np.ceil(row["Superficie efectiva [km2]"] / cobertura_km2)					# Urbano 2 km
			valores.append(dcu_requeridas)
		
	elif row["Tecnología"] == "TWACS" and row["Segmento"] == "Red":										# Caso TWACS: 1 DCU por SPD
		dcu_requeridas = row["Cantidad SPD"]							
		valores.append(dcu_requeridas)

	else:        
		dcu_necesarias = 0								# Caso de Celular y otros segmentos
		valores.append(dcu_necesarias)

df_implementacion["Cantidad DCU"] = pd.Series(valores, index=df_implementacion.index).astype('Int64')

# Se asignan 0 DCU para el resto de los años
for periodo in range(1, len(lista_dfs_implementacion)):
	lista_dfs_implementacion[periodo]["Cantidad DCU"] = 0


### Se calcula la cantidad de datos transmitida por DCU
# Se calcula la cantidad de datos transmitida por DCU (Cálulos realizados en 'Costos.xlsx')

# 1: Se consideran las exigencias de medición de la Norma SMMC hasta antes del 2034 y después de ella (Artículo 9-8)
# 0: Se consideran las exigencias de medición relajadas
exigencias_de_medicion = 0		# Input: 1 o 0 o 2

if exigencias_de_medicion == 1:
	datos_cliente_3f_zona_alta = 94.1 / 1024					# MB mensual por cliente
	datos_cliente_3f_zona_baja_previo_2034 = 45.7 / 1024		# MB mensual por cliente
	datos_cliente_1f_zona_alta = 62.8 / 1024					# MB mensual por cliente
	datos_cliente_1f_zona_baja_previo_2034 = 32.5 / 1024		# MB mensual por cliente

	datos_trafo_3f_zona_alta = 94.1 / 1024						# MB mensual por trafo
	datos_trafo_3f_zona_baja_previo_2034 = 45.7 / 1024			# MB mensual por trafo
	datos_trafo_1f_zona_alta = 72.4 / 1024						# MB mensual por trafo
	datos_trafo_1f_zona_baja_previo_2034 = 38.5 / 1024			# MB mensual por trafo

	# Valores que cambian después del 2034
	datos_cliente_3f_zona_baja_post_2034 = 94.1 / 1024			# MB mensual por cliente
	datos_cliente_1f_zona_baja_post_2034 = 62.8 / 1024			# MB mensual por cliente

	datos_trafo_3f_zona_baja_post_2034 = 94.1 / 1024			# MB mensual por trafo
	datos_trafo_1f_zona_baja_post_2034 = 72.4 / 1024			# MB mensual por trafo

else:	# Se consideran las exigencias de medición relajadas
	datos_cliente_3f_zona_alta = 38.5 / 1024					# MB mensual por cliente
	datos_cliente_3f_zona_baja_previo_2034 = 38.5 / 1024		# MB mensual por cliente
	datos_cliente_1f_zona_alta = 32.5 / 1024					# MB mensual por cliente
	datos_cliente_1f_zona_baja_previo_2034 = 32.5 / 1024		# MB mensual por cliente

	datos_trafo_3f_zona_alta = 45.7 / 1024						# MB mensual por trafo
	datos_trafo_3f_zona_baja_previo_2034 = 45.7 / 1024			# MB mensual por trafo
	datos_trafo_1f_zona_alta = 38.5 / 1024						# MB mensual por trafo
	datos_trafo_1f_zona_baja_previo_2034 = 38.5 / 1024			# MB mensual por trafo

	# Valores NO cambian después del 2034
	datos_cliente_3f_zona_baja_post_2034 = 38.5 / 1024			# MB mensual por cliente
	datos_cliente_1f_zona_baja_post_2034 = 32.5 / 1024			# MB mensual por cliente

	datos_trafo_3f_zona_baja_post_2034 = 45.7 / 1024			# MB mensual por trafo
	datos_trafo_1f_zona_baja_post_2034 = 38.5 / 1024			# MB mensual por trafo

for periodo in range(len(lista_dfs_implementacion)):
	valores = []

	if periodo < 9:		# Hasta antes del 2034

		for i, row in lista_dfs_implementacion[periodo].iloc[4::5].iterrows():			# Recorrer solo las filas de segmento "Red"		
			cantidad_dcu = lista_dfs_implementacion[0].loc[i, "Cantidad DCU"]

			# Se suman las cantidades correspondientes de UM enfocadas a clientes y transformadores según la densidad de la zona
			if cantidad_dcu > 0 and (row["Densidad"] == "ALTA" or row["Densidad"] == "MEDIA"):
				total_clientes_3f_zona_alta = lista_dfs_implementacion[periodo].loc[(row["Comuna"], ["No Residencial AT", "LibreDx"]), "Cantidad de clientes"].sum()
				total_clientes_1f_zona_alta = lista_dfs_implementacion[periodo].loc[(row["Comuna"], ["Residencial", "No Residencial BT"]), "Cantidad de clientes"].sum()

				total_trafo_3f_zona_alta = (lista_dfs_implementacion[periodo].loc[(row["Comuna"], ["Red"]), "UM 3F 1S para TD"].sum()
											+ lista_dfs_implementacion[periodo].loc[(row["Comuna"], ["Red"]), "UM 3F 2S para TD"].sum())
				total_trafo_1f_zona_alta = lista_dfs_implementacion[periodo].loc[(row["Comuna"], ["Red"]), "UM 1F para TD"].sum()

				mb_necesarios = (datos_cliente_3f_zona_alta * total_clientes_3f_zona_alta + datos_cliente_1f_zona_alta * total_clientes_1f_zona_alta
									+ datos_trafo_3f_zona_alta * total_trafo_3f_zona_alta + datos_trafo_1f_zona_alta * total_trafo_1f_zona_alta) / cantidad_dcu
				valores.append(mb_necesarios)

			elif cantidad_dcu > 0 and (row["Densidad"] == "BAJA" or row["Densidad"] == "MUY BAJA" or row["Densidad"] == "EXTREMADAMENTE BAJA"):
				total_clientes_3f_zona_baja = lista_dfs_implementacion[periodo].loc[(row["Comuna"], ["No Residencial AT", "LibreDx"]), "Cantidad de clientes"].sum()
				total_clientes_1f_zona_baja = lista_dfs_implementacion[periodo].loc[(row["Comuna"], ["Residencial", "No Residencial BT"]), "Cantidad de clientes"].sum()

				total_trafo_3f_zona_baja = (lista_dfs_implementacion[periodo].loc[(row["Comuna"], ["Red"]), "UM 3F 1S para TD"].sum()
											+ lista_dfs_implementacion[periodo].loc[(row["Comuna"], ["Red"]), "UM 3F 2S para TD"].sum())
				total_trafo_1f_zona_baja = lista_dfs_implementacion[periodo].loc[(row["Comuna"], ["Red"]), "UM 1F para TD"].sum()

				mb_necesarios = (datos_cliente_3f_zona_baja_previo_2034 * total_clientes_3f_zona_baja + datos_cliente_1f_zona_baja_previo_2034 * total_clientes_1f_zona_baja
									+ datos_trafo_3f_zona_baja_previo_2034 * total_trafo_3f_zona_baja + datos_trafo_1f_zona_baja_previo_2034 * total_trafo_1f_zona_baja) / cantidad_dcu
				valores.append(mb_necesarios)

			else:        
				mb_necesarios = 0								# Caso de Celular, TWACS y otros segmentos
				valores.append(mb_necesarios)

		lista_dfs_implementacion[periodo]["MB Mensuales por DCU"] = 0.0
		filtro = lista_dfs_implementacion[periodo]["Segmento"] == "Red"
		
		lista_dfs_implementacion[periodo].loc[filtro, "MB Mensuales por DCU"] = pd.Series(valores, index=lista_dfs_implementacion[periodo].index[filtro]).round(4)


	else:		# Después del 2034

		for i, row in lista_dfs_implementacion[periodo].iloc[4::5].iterrows():			# Recorrer solo las filas de segmento "Red"		
			cantidad_dcu = lista_dfs_implementacion[0].loc[i, "Cantidad DCU"]

			# Se suman las cantidades correspondientes de UM enfocadas a clientes y transformadores según la densidad de la zona
			if cantidad_dcu > 0 and (row["Densidad"] == "ALTA" or row["Densidad"] == "MEDIA"):
				total_clientes_3f_zona_alta = lista_dfs_implementacion[periodo].loc[(row["Comuna"], ["No Residencial AT", "LibreDx"]), "Cantidad de clientes"].sum()
				total_clientes_1f_zona_alta = lista_dfs_implementacion[periodo].loc[(row["Comuna"], ["Residencial", "No Residencial BT"]), "Cantidad de clientes"].sum()

				total_trafo_3f_zona_alta = (lista_dfs_implementacion[periodo].loc[(row["Comuna"], ["Red"]), "UM 3F 1S para TD"].sum()
											+ lista_dfs_implementacion[periodo].loc[(row["Comuna"], ["Red"]), "UM 3F 2S para TD"].sum())
				total_trafo_1f_zona_alta = lista_dfs_implementacion[periodo].loc[(row["Comuna"], ["Red"]), "UM 1F para TD"].sum()

				mb_necesarios = (datos_cliente_3f_zona_alta * total_clientes_3f_zona_alta + datos_cliente_1f_zona_alta * total_clientes_1f_zona_alta
									+ datos_trafo_3f_zona_alta * total_trafo_3f_zona_alta + datos_trafo_1f_zona_alta * total_trafo_1f_zona_alta) / cantidad_dcu
				valores.append(mb_necesarios)

			elif cantidad_dcu > 0 and (row["Densidad"] == "BAJA" or row["Densidad"] == "MUY BAJA" or row["Densidad"] == "EXTREMADAMENTE BAJA"):
				total_clientes_3f_zona_baja = lista_dfs_implementacion[periodo].loc[(row["Comuna"], ["No Residencial AT", "LibreDx"]), "Cantidad de clientes"].sum()
				total_clientes_1f_zona_baja = lista_dfs_implementacion[periodo].loc[(row["Comuna"], ["Residencial", "No Residencial BT"]), "Cantidad de clientes"].sum()

				total_trafo_3f_zona_baja = (lista_dfs_implementacion[periodo].loc[(row["Comuna"], ["Red"]), "UM 3F 1S para TD"].sum()
											+ lista_dfs_implementacion[periodo].loc[(row["Comuna"], ["Red"]), "UM 3F 2S para TD"].sum())
				total_trafo_1f_zona_baja = lista_dfs_implementacion[periodo].loc[(row["Comuna"], ["Red"]), "UM 1F para TD"].sum()

				mb_necesarios = (datos_cliente_3f_zona_baja_post_2034 * total_clientes_3f_zona_baja + datos_cliente_1f_zona_baja_post_2034 * total_clientes_1f_zona_baja
									+ datos_trafo_3f_zona_baja_post_2034 * total_trafo_3f_zona_baja + datos_trafo_1f_zona_baja_post_2034 * total_trafo_1f_zona_baja) / cantidad_dcu
				valores.append(mb_necesarios)

			else:        
				mb_necesarios = 0								# Caso de Celular, TWACS y otros segmentos
				valores.append(mb_necesarios)				

		lista_dfs_implementacion[periodo]["MB Mensuales por DCU"] = 0.0
		filtro = lista_dfs_implementacion[periodo]["Segmento"] == "Red"
		
		lista_dfs_implementacion[periodo].loc[filtro, "MB Mensuales por DCU"] = pd.Series(valores, index=lista_dfs_implementacion[periodo].index[filtro]).round(4)

## Costos de inversión y operación por comuna y segmento

In [55]:
df_capex_smi = pd.read_excel("Costos.xlsx", sheet_name="CAPEX", skiprows=4, usecols="B:M")
df_capex_smi = df_capex_smi.set_index(["Tecnología", "Densidad"], drop=False)					   # Se asignan indices

# Cálculo de la inversión
# while True:
# 	try:
# 		descuento_por_volumen = float(input("Descuento por volumen (inversión inicial) [%]: ")) / 100      # Definicion del descuento por volumen
# 		if 0 <= descuento_por_volumen <= 1:
# 			break
# 	except ValueError:
# 		pass

descuento_por_volumen = 0.20      # Definicion del descuento por volumen

for periodo in range(len(lista_dfs_implementacion)):

	inversion_um_1f = []
	inversion_um_3f_dir = []
	inversion_um_3f_ind = []
	inversion_dcu = []
	inversion_sgo = []
	inversion_mdms = []

	for i, row in lista_dfs_implementacion[periodo].iterrows():
		tecnologia = row["Tecnología"]
		densidad = row["Densidad"]

		costo_um_1f = (df_capex_smi.loc[(tecnologia, densidad), "UM 1F [USD/u]"] * (1 - descuento_por_volumen))
		costo_um_3f_dir = (df_capex_smi.loc[(tecnologia, densidad), "UM 3F Directa [USD/u]"] * (1 - descuento_por_volumen))
		costo_um_3f_ind = (df_capex_smi.loc[(tecnologia, densidad), "UM 3F Indirecta [USD/u]"] * (1 - descuento_por_volumen))
		costo_dcu = (df_capex_smi.loc[(tecnologia, densidad), "DCU [USD/u]"] * (1 - descuento_por_volumen))
		costo_sgo = df_capex_smi.loc[(tecnologia, densidad), "SGO [USD/Medidor]"]
		costo_mdms = df_capex_smi.loc[(tecnologia, densidad), "MDMS [USD/Medidor]"]

		instalacion_um_1f = (df_capex_smi.loc[(tecnologia, densidad), "Instalación UM 1F [USD/u]"] * (1 - descuento_por_volumen))
		instalacion_um_3f_dir = (df_capex_smi.loc[(tecnologia, densidad), "Instalación UM 3F Directa [USD/u]"] * (1 - descuento_por_volumen))
		instalacion_um_3f_ind = (df_capex_smi.loc[(tecnologia, densidad), "Instalación UM 3F Indirecta [USD/u]"] * (1 - descuento_por_volumen))
		instalacion_dcu = (df_capex_smi.loc[(tecnologia, densidad), "Instalación DCU [USD/u]"] * (1 - descuento_por_volumen))

		inv_um_1f = (row["Cantidad UM 1F"] * costo_um_1f + row["Cantidad UM 1F"] * instalacion_um_1f)
		inv_um_3f_dir = (row["Cantidad UM 3F Directa"] * costo_um_3f_dir + row["Cantidad UM 3F Directa"] * instalacion_um_3f_dir)
		inv_um_3f_ind = (row["Cantidad UM 3F Indirecta"] * costo_um_3f_ind + row["Cantidad UM 3F Indirecta"] * instalacion_um_3f_ind)
		inv_dcu = (row["Cantidad DCU"] * costo_dcu + row["Cantidad DCU"] * instalacion_dcu)
		inv_sgo = ((row["Cantidad UM 1F"] + row["Cantidad UM 3F Directa"] + row["Cantidad UM 3F Indirecta"]) * costo_sgo)
		inv_mdms = ((row["Cantidad UM 1F"] + row["Cantidad UM 3F Directa"] + row["Cantidad UM 3F Indirecta"]) * costo_mdms)

		inversion_um_1f.append(inv_um_1f)
		inversion_um_3f_dir.append(inv_um_3f_dir)
		inversion_um_3f_ind.append(inv_um_3f_ind)
		inversion_dcu.append(inv_dcu)
		inversion_sgo.append(inv_sgo)
		inversion_mdms.append(inv_mdms)

	# Se agregan las columnas de inversión por equipo al DataFrame
	lista_dfs_implementacion[periodo]["Inversión UM 1F [USD]"] = pd.Series(inversion_um_1f, index=lista_dfs_implementacion[periodo].index)
	lista_dfs_implementacion[periodo]["Inversión UM 3F Directa [USD]"] = pd.Series(inversion_um_3f_dir, index=lista_dfs_implementacion[periodo].index)
	lista_dfs_implementacion[periodo]["Inversión UM 3F Indirecta [USD]"] = pd.Series(inversion_um_3f_ind, index=lista_dfs_implementacion[periodo].index)
	lista_dfs_implementacion[periodo]["Inversión DCU [USD]"] = pd.Series(inversion_dcu, index=lista_dfs_implementacion[periodo].index)
	lista_dfs_implementacion[periodo]["Inversión SGO [USD]"] = pd.Series(inversion_sgo, index=lista_dfs_implementacion[periodo].index)
	lista_dfs_implementacion[periodo]["Inversión MDMS [USD]"] = pd.Series(inversion_mdms, index=lista_dfs_implementacion[periodo].index)

	# Se suma la inversión total por comuna y segmento
	lista_dfs_implementacion[periodo]["Inversión Total [USD]"] = (lista_dfs_implementacion[periodo]["Inversión UM 1F [USD]"]
																	+ lista_dfs_implementacion[periodo]["Inversión UM 3F Directa [USD]"]
																	  + lista_dfs_implementacion[periodo]["Inversión UM 3F Indirecta [USD]"]
																	 + lista_dfs_implementacion[periodo]["Inversión DCU [USD]"]
																	+ lista_dfs_implementacion[periodo]["Inversión SGO [USD]"]
																	+ lista_dfs_implementacion[periodo]["Inversión MDMS [USD]"]).round(4)


### Se crea un nuevo DataFrame para almacenar los resultados finales
df_resultado_final = pd.DataFrame()
df_resultado_final_alta = pd.DataFrame()
df_resultado_final_media = pd.DataFrame()
df_resultado_final_baja = pd.DataFrame()
df_resultado_final_muy_baja = pd.DataFrame()
df_resultado_final_ext_baja = pd.DataFrame()

inversion = [
	"Energía facturada anual [MWh/año]",
	"Cantidad UM 1F",
	"Cantidad UM 3F Directa",
	"Cantidad UM 3F Indirecta",
	"Cantidad DCU",
	"Inversión UM 1F [USD/equipo]",
	"Inversión UM 3F Directa [USD/equipo]",
	"Inversión UM 3F Indirecta [USD/equipo]",
	"Inversión DCU [USD/equipo]",
	"Inversión SGO [USD]",
	"Inversión MDMS [USD]"
]

df_resultado_final["Concepto"] = inversion
df_resultado_final_alta["Concepto"] = inversion
df_resultado_final_media["Concepto"] = inversion
df_resultado_final_baja["Concepto"] = inversion
df_resultado_final_muy_baja["Concepto"] = inversion
df_resultado_final_ext_baja["Concepto"] = inversion

for periodo in range(len(lista_dfs_implementacion)):
	
	# Filtrar DataFrame por densidad
	df_final = lista_dfs_implementacion[periodo]
	df_alta = lista_dfs_implementacion[periodo][lista_dfs_implementacion[periodo]["Densidad"] == "ALTA"]
	df_media = lista_dfs_implementacion[periodo][lista_dfs_implementacion[periodo]["Densidad"] == "MEDIA"]
	df_baja = lista_dfs_implementacion[periodo][lista_dfs_implementacion[periodo]["Densidad"] == "BAJA"]
	df_muy_baja = lista_dfs_implementacion[periodo][lista_dfs_implementacion[periodo]["Densidad"] == "MUY BAJA"]
	df_ext_baja = lista_dfs_implementacion[periodo][lista_dfs_implementacion[periodo]["Densidad"] == "EXTREMADAMENTE BAJA"]
	
	# Se agregan las inversiones de cada item por periodo
	nuevo_periodo = [
		df_final["Energía facturada anual [MWh/año]"].sum(),
		df_final["Cantidad UM 1F"].sum(),
		df_final["Cantidad UM 3F Directa"].sum(),
		df_final["Cantidad UM 3F Indirecta"].sum(),
		df_final["Cantidad DCU"].sum(),
		df_final["Inversión UM 1F [USD]"].sum() / df_final["Cantidad UM 1F"].sum() if df_final["Cantidad UM 1F"].sum() > 0 else 0,
		df_final["Inversión UM 3F Directa [USD]"].sum() / df_final["Cantidad UM 3F Directa"].sum() if df_final["Cantidad UM 3F Directa"].sum() > 0 else 0,
		df_final["Inversión UM 3F Indirecta [USD]"].sum() / df_final["Cantidad UM 3F Indirecta"].sum() if df_final["Cantidad UM 3F Indirecta"].sum() > 0 else 0,
		df_final["Inversión DCU [USD]"].sum() / df_final["Cantidad DCU"].sum() if df_final["Cantidad DCU"].sum() > 0 else 0,
		df_final["Inversión SGO [USD]"].sum(),
		df_final["Inversión MDMS [USD]"].sum()
	]

	# Se agregan las inversiones de cada item por periodo
	nuevo_periodo_alta = [
		df_alta["Energía facturada anual [MWh/año]"].sum(),
		df_alta["Cantidad UM 1F"].sum(),
		df_alta["Cantidad UM 3F Directa"].sum(),
		df_alta["Cantidad UM 3F Indirecta"].sum(),
		df_alta["Cantidad DCU"].sum(),
		df_alta["Inversión UM 1F [USD]"].sum() / df_alta["Cantidad UM 1F"].sum() if df_alta["Cantidad UM 1F"].sum() > 0 else 0,
		df_alta["Inversión UM 3F Directa [USD]"].sum() / df_alta["Cantidad UM 3F Directa"].sum() if df_alta["Cantidad UM 3F Directa"].sum() > 0 else 0,
		df_alta["Inversión UM 3F Indirecta [USD]"].sum() / df_alta["Cantidad UM 3F Indirecta"].sum() if df_alta["Cantidad UM 3F Indirecta"].sum() > 0 else 0,
		df_alta["Inversión DCU [USD]"].sum() / df_alta["Cantidad DCU"].sum() if df_alta["Cantidad DCU"].sum() > 0 else 0,
		df_alta["Inversión SGO [USD]"].sum(),
		df_alta["Inversión MDMS [USD]"].sum()
	]

	# Se agregan las inversiones de cada item por periodo
	nuevo_periodo_media = [
		df_media["Energía facturada anual [MWh/año]"].sum(),
		df_media["Cantidad UM 1F"].sum(),
		df_media["Cantidad UM 3F Directa"].sum(),
		df_media["Cantidad UM 3F Indirecta"].sum(),
		df_media["Cantidad DCU"].sum(),
		df_media["Inversión UM 1F [USD]"].sum() / df_media["Cantidad UM 1F"].sum() if df_media["Cantidad UM 1F"].sum() > 0 else 0,
		df_media["Inversión UM 3F Directa [USD]"].sum() / df_media["Cantidad UM 3F Directa"].sum() if df_media["Cantidad UM 3F Directa"].sum() > 0 else 0,
		df_media["Inversión UM 3F Indirecta [USD]"].sum() / df_media["Cantidad UM 3F Indirecta"].sum() if df_media["Cantidad UM 3F Indirecta"].sum() > 0 else 0,
		df_media["Inversión DCU [USD]"].sum() / df_media["Cantidad DCU"].sum() if df_media["Cantidad DCU"].sum() > 0 else 0,
		df_media["Inversión SGO [USD]"].sum(),
		df_media["Inversión MDMS [USD]"].sum()
	]

	# Se agregan las inversiones de cada item por periodo
	nuevo_periodo_baja = [
		df_baja["Energía facturada anual [MWh/año]"].sum(),
		df_baja["Cantidad UM 1F"].sum(),
		df_baja["Cantidad UM 3F Directa"].sum(),
		df_baja["Cantidad UM 3F Indirecta"].sum(),
		df_baja["Cantidad DCU"].sum(),
		df_baja["Inversión UM 1F [USD]"].sum() / df_baja["Cantidad UM 1F"].sum() if df_baja["Cantidad UM 1F"].sum() > 0 else 0,
		df_baja["Inversión UM 3F Directa [USD]"].sum() / df_baja["Cantidad UM 3F Directa"].sum() if df_baja["Cantidad UM 3F Directa"].sum() > 0 else 0,
		df_baja["Inversión UM 3F Indirecta [USD]"].sum() / df_baja["Cantidad UM 3F Indirecta"].sum() if df_baja["Cantidad UM 3F Indirecta"].sum() > 0 else 0,
		df_baja["Inversión DCU [USD]"].sum() / df_baja["Cantidad DCU"].sum() if df_baja["Cantidad DCU"].sum() > 0 else 0,
		df_baja["Inversión SGO [USD]"].sum(),
		df_baja["Inversión MDMS [USD]"].sum()
	]

	# Se agregan las inversiones de cada item por periodo
	nuevo_periodo_muy_baja = [
		df_muy_baja["Energía facturada anual [MWh/año]"].sum(),
		df_muy_baja["Cantidad UM 1F"].sum(),
		df_muy_baja["Cantidad UM 3F Directa"].sum(),
		df_muy_baja["Cantidad UM 3F Indirecta"].sum(),
		df_muy_baja["Cantidad DCU"].sum(),
		df_muy_baja["Inversión UM 1F [USD]"].sum() / df_muy_baja["Cantidad UM 1F"].sum() if df_muy_baja["Cantidad UM 1F"].sum() > 0 else 0,
		df_muy_baja["Inversión UM 3F Directa [USD]"].sum() / df_muy_baja["Cantidad UM 3F Directa"].sum() if df_muy_baja["Cantidad UM 3F Directa"].sum() > 0 else 0,
		df_muy_baja["Inversión UM 3F Indirecta [USD]"].sum() / df_muy_baja["Cantidad UM 3F Indirecta"].sum() if df_muy_baja["Cantidad UM 3F Indirecta"].sum() > 0 else 0,
		df_muy_baja["Inversión DCU [USD]"].sum() / df_muy_baja["Cantidad DCU"].sum() if df_muy_baja["Cantidad DCU"].sum() > 0 else 0,
		df_muy_baja["Inversión SGO [USD]"].sum(),
		df_muy_baja["Inversión MDMS [USD]"].sum()
	]

	# Se agregan las inversiones de cada item por periodo
	nuevo_periodo_ext_baja = [
		df_ext_baja["Energía facturada anual [MWh/año]"].sum(),
		df_ext_baja["Cantidad UM 1F"].sum(),
		df_ext_baja["Cantidad UM 3F Directa"].sum(),
		df_ext_baja["Cantidad UM 3F Indirecta"].sum(),
		df_ext_baja["Cantidad DCU"].sum(),
		df_ext_baja["Inversión UM 1F [USD]"].sum() / df_ext_baja["Cantidad UM 1F"].sum() if df_ext_baja["Cantidad UM 1F"].sum() > 0 else 0,
		df_ext_baja["Inversión UM 3F Directa [USD]"].sum() / df_ext_baja["Cantidad UM 3F Directa"].sum() if df_ext_baja["Cantidad UM 3F Directa"].sum() > 0 else 0,
		df_ext_baja["Inversión UM 3F Indirecta [USD]"].sum() / df_ext_baja["Cantidad UM 3F Indirecta"].sum() if df_ext_baja["Cantidad UM 3F Indirecta"].sum() > 0 else 0,
		df_ext_baja["Inversión DCU [USD]"].sum() / df_ext_baja["Cantidad DCU"].sum() if df_ext_baja["Cantidad DCU"].sum() > 0 else 0,
		df_ext_baja["Inversión SGO [USD]"].sum(),
		df_ext_baja["Inversión MDMS [USD]"].sum()
	]

	df_resultado_final["Periodo {}".format(periodo)] = nuevo_periodo
	df_resultado_final_alta["Periodo {}".format(periodo)] = nuevo_periodo_alta
	df_resultado_final_media["Periodo {}".format(periodo)] = nuevo_periodo_media
	df_resultado_final_baja["Periodo {}".format(periodo)] = nuevo_periodo_baja
	df_resultado_final_muy_baja["Periodo {}".format(periodo)] = nuevo_periodo_muy_baja
	df_resultado_final_ext_baja["Periodo {}".format(periodo)] = nuevo_periodo_ext_baja
	
	lista_dfs_implementacion[periodo].drop(columns=[#"Inversión UM 1F [USD]", "Inversión UM 3F Directa [USD]", 
													#"Inversión UM 3F Indirecta [USD]", "Inversión DCU [USD]", 
													"Inversión SGO [USD]", "Inversión MDMS [USD]"], inplace=True)
	

### Costos de Operacrion y Mantenimiento (O&M)
df_opex_smi = pd.read_excel("Costos.xlsx", sheet_name="OPEX", skiprows=4, usecols="B:G")
df_opex_smi = df_opex_smi.set_index(["Tecnología", "Densidad"], drop=False)			        # Se asignan indices

# Planes de datos, cotización de entel a Dic 2022 (ajustado por IPC, factor=1.12)
plan_10_mb = (1200 / usd) * 1.12  			# Costo mensual en pesos chilenos para 10 MB
plan_10_mb_extra = (100 / usd) * 1.12  		# Costo mensual en pesos chilenos por cada 1 MB extra sobre el plan base 10 MB
plan_200_mb = (2800 / usd) * 1.12  			# Costo mensual en pesos chilenos para 200 MB
plan_300_mb = (3300 / usd) * 1.12  			# Costo mensual en pesos chilenos para 300 MB
plan_200_300_mb_extra = (50 / usd) * 1.12  	# Costo mensual en pesos chilenos por cada 1 MB extra sobre el plan base 200 MB o 300 MB

for periodo in range(len(lista_dfs_implementacion)):

	# Cálculo de los costos de operación
	comunicaciones = []
	comunicaciones_dcu_sgo = []
	sgo = []
	mdms = []
	mantenimiento_equipos = []

	for i, row in lista_dfs_implementacion[periodo].iterrows():
		tecnologia = row["Tecnología"]
		densidad = row["Densidad"]

		opex_com = df_opex_smi.loc[(tecnologia, densidad), "Comunicaciones [USD/medidor]"] 
		opex_sgo = df_opex_smi.loc[(tecnologia, densidad), "Operación SGO [USD/medidor]"]
		opex_mdms = df_opex_smi.loc[(tecnologia, densidad), "Operación MDMS [USD/medidor]"]
		opex_mantenimiento_equipos = df_opex_smi.loc[(tecnologia, densidad), "Soporte y Mantenimiento Equipos [USD/medidor]"]

		operacion_com = ((row["Cantidad UM 1F"] + row["Cantidad UM 3F Directa"] + row["Cantidad UM 3F Indirecta"]) * opex_com)
		operacion_sgo = ((row["Cantidad UM 1F"] + row["Cantidad UM 3F Directa"] + row["Cantidad UM 3F Indirecta"]) * opex_sgo)
		operacion_mdms = ((row["Cantidad UM 1F"] + row["Cantidad UM 3F Directa"] + row["Cantidad UM 3F Indirecta"]) * opex_mdms)
		operacion_mantenimiento_equipos = ((row["Cantidad UM 1F"] + row["Cantidad UM 3F Directa"] + row["Cantidad UM 3F Indirecta"]) * opex_mantenimiento_equipos)

		cantidad_dcu = df_implementacion.loc[i, "Cantidad DCU"]
		mb_mensuales = row["MB Mensuales por DCU"]

		if mb_mensuales < 26:                   # Conveniencia económica de plan 10 MB
			operacion_com_dcu_sgo = ((plan_10_mb + round(mb_mensuales - 10) * plan_10_mb_extra) * cantidad_dcu * 12) if mb_mensuales >= 10 else (plan_10_mb * cantidad_dcu * 12)
			comunicaciones_dcu_sgo.append(operacion_com_dcu_sgo)
		if 26 <= mb_mensuales <= 210:			# Conveniencia económica de plan 200 MB
			operacion_com_dcu_sgo = ((plan_200_mb + round(mb_mensuales - 200) * plan_200_300_mb_extra) * cantidad_dcu * 12) if mb_mensuales > 200 else (plan_200_mb * cantidad_dcu * 12)
			comunicaciones_dcu_sgo.append(operacion_com_dcu_sgo)
		if 210 < mb_mensuales:
			operacion_com_dcu_sgo = ((plan_300_mb + round(mb_mensuales - 300) * plan_200_300_mb_extra) * cantidad_dcu * 12) if mb_mensuales > 300 else (plan_300_mb * cantidad_dcu * 12)
			comunicaciones_dcu_sgo.append(operacion_com_dcu_sgo)

		comunicaciones.append(operacion_com)
		sgo.append(operacion_sgo)
		mdms.append(operacion_mdms)
		mantenimiento_equipos.append(operacion_mantenimiento_equipos)

	# Se agregan las columnas de inversión por equipo al DataFrame
	lista_dfs_implementacion[periodo]["Operación Comunicaciones [USD/año]"] = pd.Series(comunicaciones, index=lista_dfs_implementacion[periodo].index)
	lista_dfs_implementacion[periodo]["Operación Comunicaciones DCU-SGO [USD/año]"] = pd.Series(comunicaciones_dcu_sgo, index=lista_dfs_implementacion[periodo].index).astype("Float64")
	lista_dfs_implementacion[periodo]["Operación SGO [USD/año]"] = pd.Series(sgo, index=lista_dfs_implementacion[periodo].index)
	lista_dfs_implementacion[periodo]["Operación MDMS [USD/año]"] = pd.Series(mdms, index=lista_dfs_implementacion[periodo].index)
	lista_dfs_implementacion[periodo]["Soporte y Mantenimiento Equipos [USD/año]"] = pd.Series(mantenimiento_equipos, index=lista_dfs_implementacion[periodo].index)

	if 0 < periodo <= len(lista_dfs_implementacion):
		lista_dfs_implementacion[periodo]["Operación Comunicaciones [USD/año]"] += lista_dfs_implementacion[periodo - 1]["Operación Comunicaciones [USD/año]"]
		lista_dfs_implementacion[periodo]["Operación SGO [USD/año]"] += lista_dfs_implementacion[periodo - 1]["Operación SGO [USD/año]"]
		lista_dfs_implementacion[periodo]["Operación MDMS [USD/año]"] += lista_dfs_implementacion[periodo - 1]["Operación MDMS [USD/año]"]
		lista_dfs_implementacion[periodo]["Soporte y Mantenimiento Equipos [USD/año]"] += lista_dfs_implementacion[periodo - 1]["Soporte y Mantenimiento Equipos [USD/año]"]

	# Se suma la inversión total por comuna y segmento
	lista_dfs_implementacion[periodo]["Costos Operacionales Totales [USD/año]"] = (lista_dfs_implementacion[periodo]["Operación Comunicaciones [USD/año]"]
																					+ lista_dfs_implementacion[periodo]["Operación Comunicaciones DCU-SGO [USD/año]"]
																					+ lista_dfs_implementacion[periodo]["Operación SGO [USD/año]"]
																					+ lista_dfs_implementacion[periodo]["Operación MDMS [USD/año]"]
																					+ lista_dfs_implementacion[periodo]["Soporte y Mantenimiento Equipos [USD/año]"]).round(4)
	

### Se crea un nuevo DataFrame para almacenar los resultados finales de OPEX
df_resultado_final_opex = pd.DataFrame()
df_resultado_final_opex_alta = pd.DataFrame()
df_resultado_final_opex_media = pd.DataFrame()
df_resultado_final_opex_baja = pd.DataFrame()
df_resultado_final_opex_muy_baja = pd.DataFrame()
df_resultado_final_opex_ext_baja = pd.DataFrame()

operacion = [
	"Operación Comunicaciones [USD/año]",
	"Operación Comunicaciones DCU-SGO [USD/año]",
	"Operación SGO [USD/año]",
	"Operación MDMS [USD/año]",
	"Soporte y Mantenimiento Equipos [USD/año]"
]

df_resultado_final_opex["Concepto"] = operacion
df_resultado_final_opex_alta["Concepto"] = operacion
df_resultado_final_opex_media["Concepto"] = operacion
df_resultado_final_opex_baja["Concepto"] = operacion
df_resultado_final_opex_muy_baja["Concepto"] = operacion
df_resultado_final_opex_ext_baja["Concepto"] = operacion

for periodo in range(len(lista_dfs_implementacion)):
	
	# Filtrar DataFrame por densidad
	df_final = lista_dfs_implementacion[periodo]
	df_alta = lista_dfs_implementacion[periodo][lista_dfs_implementacion[periodo]["Densidad"] == "ALTA"]
	df_media = lista_dfs_implementacion[periodo][lista_dfs_implementacion[periodo]["Densidad"] == "MEDIA"]
	df_baja = lista_dfs_implementacion[periodo][lista_dfs_implementacion[periodo]["Densidad"] == "BAJA"]
	df_muy_baja = lista_dfs_implementacion[periodo][lista_dfs_implementacion[periodo]["Densidad"] == "MUY BAJA"]
	df_ext_baja = lista_dfs_implementacion[periodo][lista_dfs_implementacion[periodo]["Densidad"] == "EXTREMADAMENTE BAJA"]
	
	# Se agregan los costos operacionales de cada item por periodo
	nuevo_periodo_opex = [
		df_final["Operación Comunicaciones [USD/año]"].sum(),
		df_final["Operación Comunicaciones DCU-SGO [USD/año]"].sum(),
		df_final["Operación SGO [USD/año]"].sum(),
		df_final["Operación MDMS [USD/año]"].sum(),
		df_final["Soporte y Mantenimiento Equipos [USD/año]"].sum()
	]

	# Se agregan los costos operacionales de cada item por periodo
	nuevo_periodo_opex_alta = [
		df_alta["Operación Comunicaciones [USD/año]"].sum(),
		df_alta["Operación Comunicaciones DCU-SGO [USD/año]"].sum(),
		df_alta["Operación SGO [USD/año]"].sum(),
		df_alta["Operación MDMS [USD/año]"].sum(),
		df_alta["Soporte y Mantenimiento Equipos [USD/año]"].sum()
	]

	nuevo_periodo_opex_media = [
		df_media["Operación Comunicaciones [USD/año]"].sum(),
		df_media["Operación Comunicaciones DCU-SGO [USD/año]"].sum(),
		df_media["Operación SGO [USD/año]"].sum(),
		df_media["Operación MDMS [USD/año]"].sum(),
		df_media["Soporte y Mantenimiento Equipos [USD/año]"].sum()
	]

	nuevo_periodo_opex_baja = [
		df_baja["Operación Comunicaciones [USD/año]"].sum(),
		df_baja["Operación Comunicaciones DCU-SGO [USD/año]"].sum(),
		df_baja["Operación SGO [USD/año]"].sum(),
		df_baja["Operación MDMS [USD/año]"].sum(),
		df_baja["Soporte y Mantenimiento Equipos [USD/año]"].sum()
	]

	nuevo_periodo_opex_muy_baja = [
		df_muy_baja["Operación Comunicaciones [USD/año]"].sum(),
		df_muy_baja["Operación Comunicaciones DCU-SGO [USD/año]"].sum(),
		df_muy_baja["Operación SGO [USD/año]"].sum(),
		df_muy_baja["Operación MDMS [USD/año]"].sum(),
		df_muy_baja["Soporte y Mantenimiento Equipos [USD/año]"].sum()
	]

	nuevo_periodo_opex_ext_baja = [
		df_ext_baja["Operación Comunicaciones [USD/año]"].sum(),
		df_ext_baja["Operación Comunicaciones DCU-SGO [USD/año]"].sum(),
		df_ext_baja["Operación SGO [USD/año]"].sum(),
		df_ext_baja["Operación MDMS [USD/año]"].sum(),
		df_ext_baja["Soporte y Mantenimiento Equipos [USD/año]"].sum()
	]

	df_resultado_final_opex["Periodo {}".format(periodo)] = nuevo_periodo_opex
	df_resultado_final_opex_alta["Periodo {}".format(periodo)] = nuevo_periodo_opex_alta
	df_resultado_final_opex_media["Periodo {}".format(periodo)] = nuevo_periodo_opex_media
	df_resultado_final_opex_baja["Periodo {}".format(periodo)] = nuevo_periodo_opex_baja
	df_resultado_final_opex_muy_baja["Periodo {}".format(periodo)] = nuevo_periodo_opex_muy_baja
	df_resultado_final_opex_ext_baja["Periodo {}".format(periodo)] = nuevo_periodo_opex_ext_baja

	lista_dfs_implementacion[periodo].drop(columns=["Operación Comunicaciones [USD/año]", "Operación Comunicaciones DCU-SGO [USD/año]", 
													  "Operación SGO [USD/año]", "Operación MDMS [USD/año]", 
													  "Soporte y Mantenimiento Equipos [USD/año]"], inplace=True)
	
df_resultado_final = pd.concat([df_resultado_final, df_resultado_final_opex], ignore_index=True)
df_resultado_final_alta = pd.concat([df_resultado_final_alta, df_resultado_final_opex_alta], ignore_index=True)
df_resultado_final_media = pd.concat([df_resultado_final_media, df_resultado_final_opex_media], ignore_index=True)
df_resultado_final_baja = pd.concat([df_resultado_final_baja, df_resultado_final_opex_baja], ignore_index=True)
df_resultado_final_muy_baja = pd.concat([df_resultado_final_muy_baja, df_resultado_final_opex_muy_baja], ignore_index=True)
df_resultado_final_ext_baja = pd.concat([df_resultado_final_ext_baja, df_resultado_final_opex_ext_baja], ignore_index=True)


# Inversión y Operación total por periodo
print("\nConsiderando un descuento por volumen de compra e instalación de UM: {:,.0f}%".format(descuento_por_volumen * 100))
print("----------------------------------------------------------------------------------------")
print("\033[1mPeriodo 0)\033[0m", "Inversión total: M USD {:,.4f}".format(lista_dfs_implementacion[0]["Inversión Total [USD]"].sum() / 1_000_000), 
		  " | Costo Operacional total: M USD {:,.4f}".format(lista_dfs_implementacion[0]["Costos Operacionales Totales [USD/año]"].sum() / 1_000_000))
print("----------------------------------------------------------------------------------------")
for periodo in range(1, len(lista_dfs_implementacion)):
	print("-> \033[1mPeriodo {})\033[0m".format(periodo), "Inversión total: M USD {:,.4f}".format(lista_dfs_implementacion[periodo]["Inversión Total [USD]"].sum() / 1_000_000), 
		  " | Costo Operacional total: M USD {:,.4f}".format(lista_dfs_implementacion[periodo]["Costos Operacionales Totales [USD/año]"].sum() / 1_000_000))


Considerando un descuento por volumen de compra e instalación de UM: 20%
----------------------------------------------------------------------------------------
Periodo 0) Inversión total: M USD 1,563.1932  | Costo Operacional total: M USD 83.6617
----------------------------------------------------------------------------------------
-> Periodo 1) Inversión total: M USD 20.3231  | Costo Operacional total: M USD 84.8544
-> Periodo 2) Inversión total: M USD 20.6299  | Costo Operacional total: M USD 86.0655
-> Periodo 3) Inversión total: M USD 20.9452  | Costo Operacional total: M USD 87.2953
-> Periodo 4) Inversión total: M USD 21.2648  | Costo Operacional total: M USD 88.5441
-> Periodo 5) Inversión total: M USD 21.5956  | Costo Operacional total: M USD 89.8124
-> Periodo 6) Inversión total: M USD 21.9234  | Costo Operacional total: M USD 91.1003
-> Periodo 7) Inversión total: M USD 22.2616  | Costo Operacional total: M USD 92.4082
-> Periodo 8) Inversión total: M USD 22.5973  | Cost

## Beneficios esperados por Comuna y Segmento

In [56]:
### Algunas consideraciones y supuestos para el cálculo de los beneficios:

# PORCENTAJES DE REDUCCIÓN (inputs para el cálculo de beneficios) (Seguros, NO MODIFICAR)
reduccion_lectura_pedestre 		  = 1.000			# 100%
reduccion_corte_y_repo 			  = 0.810			# ~81%

reduccion_duracion_falla_alta 	  = 0.050			# 5% - 35%
reduccion_duracion_falla_media 	  = 0.100			# 5% - 35%
reduccion_duracion_falla_baja 	  = 0.200			# 5% - 35%
reduccion_duracion_falla_muy_baja = 0.300			# 5% - 35%
reduccion_duracion_falla_ext_baja = 0.350			# 5% - 35%

reduccion_atencion_clientes 	  = 0.600			# ~60%
reduccion_infraestructura 	 	  = 0.075			# 5% - 10%
reduccion_pnt_hurto 			  = 0.700			# 50% - 80%
reduccion_pnt_comercial 		  = 0.700			# 40% - 60%

# PORCENTAJES DE REDUCCIÓN (inputs para el cálculo de beneficios) (Inciertos, MODIFICAR SEGÚN ESCENARIO)
# reduccion_precio_de_la_energia 	  = 0.100			# 0% - 79%
reduccion_consumo_residencial 	  = 0.085			# 3% - 20%     (Reducción solo aplicable al 7,5% de todos los clientes)
reduccion_consumo_no_residencial  = 0.060			# 2,8% - 15%   (Reducción solo aplicable al 7,5% de todos los clientes)
# En el escenario base, la reducción del precio de la energía debe ser del 59% para un VAN = 0. 
# En el escenario pesimista, la reducción del precio de la energía debe ser del 72,7% para un VAN = 0.
# En el escenario optimista, la reducción del precio de la energía debe ser del 45,1% para un VAN = 0.

#---------------------------------------------------------------------------------------

# Precio de la energía (PNP equivalente nacional, Anexos PNP Julio 2025)
precio_energia_usd_mwh = 93.14   		# USD/MWh

# Compensaciones automáticas e instruidas por la SEC por fallas en el suministro eléctrico (Fuente: Informe SEC Dic 2024)
monto_compensado_anual = (8_744_000_000 + 3_961_000_000) * 1.037 / usd			# USD/año Automáticas + Instruidas (2024)
cantidad_de_compensaciones = 3_257_118 + 1_150_658								# N° Automáticas + Instruidas (2024)

# Costo de falla 2024 para el SEN bajo una profundidad de entre 0-5%
costo_de_falla_2024 = 415.60			# USD/MWh

# Cálculo de la energía no suministrada anual (fuente SEC)
energia_no_suministrada_anual = monto_compensado_anual / (2 * costo_de_falla_2024)	 	# MWh/año


### Beneficio 1) Se calcula el ahorro anual por reducción en la lectura pedestre

# Costo de lectura pedestre por medición (Datos del proceso VAD 2024-2028, 'Beneficios.xlsx')
costos_por_densidad = {
	"ALTA": 0.2042,						# USD/medición
	"MEDIA": 0.2653,					# USD/medición
	"BAJA": 0.4133,					    # USD/medición
	"MUY BAJA": 0.7136,					# USD/medición
	"EXTREMADAMENTE BAJA": 0.8853		# USD/medición	
}

for periodo in range(len(lista_dfs_implementacion)):

	# Se considera entonces:
	# Medición mensual: costo_lectura * 12
	# Medición bimensual: costo_lectura * 6

	# Costo base por registro (por medición)
	costo_base = lista_dfs_implementacion[periodo]["Densidad"].map(costos_por_densidad)

	# Factor anual según periodicidad: mensual = 12, bimensual = 6
	factor_anual = np.where(lista_dfs_implementacion[periodo]["Tipo de facturación"] == "Mensual", 12,
					np.where(lista_dfs_implementacion[periodo]["Tipo de facturación"] == "Bimensual", 6, np.nan))
	
	# Indice para aplicar el cálculo solo al segmento Regulado (Residencial y No Residencial)
	indice_segmento = np.where(lista_dfs_implementacion[periodo]["Segmento"] == "Residencial", 1,
						np.where(lista_dfs_implementacion[periodo]["Segmento"] == "No Residencial BT", 1,
							   np.where(lista_dfs_implementacion[periodo]["Segmento"] == "No Residencial AT", 1, 0)))

	# Nueva columna con costo anual
	lista_dfs_implementacion[periodo]["Ahorro en lectura pedestre [USD/año]"] = (costo_base
																				* factor_anual
																				* lista_dfs_implementacion[periodo]["Cantidad de clientes"]
																				* indice_segmento
																				* reduccion_lectura_pedestre).round(4)


### Beneficio 2) Ahorros asociados al corte y reposición remotos de suministro

# Costo promedio por corte e reposición de suministro (Datos del proceso VAD 2024-2028, 'Beneficios.xlsx')
corte_y_repo = [0.4534, 0.4534, 0.4534, 0.4534, 0.4534, 0.4534, 0.4534, 0.4534, 
				   0.4534, 0.4534, 0.4534, 0.4534, 0.4534, 0.4537, 0.4540, 0.4543]				# USD/cliente-año por año desde 2025 a 2040

for periodo in range(len(lista_dfs_implementacion)):

	# Indice para aplicar el cálculo solo al segmento Regulado (Residencial y No_Residencial)
	indice_segmento = np.where(lista_dfs_implementacion[periodo]["Segmento"] == "Residencial", 1,
						np.where(lista_dfs_implementacion[periodo]["Segmento"] == "No Residencial BT", 1,
							   np.where(lista_dfs_implementacion[periodo]["Segmento"] == "No Residencial AT", 1, 0)))

	# Ahorro anual por reducción en cortes y reposición remotos de suministro
	lista_dfs_implementacion[periodo]["Ahorro corte y reposición remoto [USD/año]"] = (lista_dfs_implementacion[periodo]["Cantidad de clientes"]
																						* corte_y_repo[periodo]
																						* reduccion_corte_y_repo
																						* indice_segmento)
	

### Beneficio 3) Reducción en el tiempo sin suministro eléctrico (Datos del proceso VAD 2024-2028, 'Beneficios.xlsx')
# cantidad_fallas_anual = 56855     # Fallas/año
# duracion_promedio_anual = 5.62    # Horas/falla

# Se busca la energía no suministrada total desagregada por nivel de densidad
energia_por_densidad = df_implementacion.groupby("Densidad")["Energía facturada anual [MWh/año]"].sum()
energia_total = energia_por_densidad.sum()

proporcion_energia_alta = energia_por_densidad["ALTA"] / energia_total
proporcion_energia_media = energia_por_densidad["MEDIA"] / energia_total
proporcion_energia_baja = energia_por_densidad["BAJA"] / energia_total
proporcion_energia_muy_baja = energia_por_densidad["MUY BAJA"] / energia_total
proporcion_energia_extremadamente_baja = energia_por_densidad["EXTREMADAMENTE BAJA"] / energia_total

duracion_fallas_anuales_alta = 0.0124
duracion_fallas_anuales_media = 0.6898
duracion_fallas_anuales_baja = 0.2769
duracion_fallas_anuales_muy_baja = 0.0114
duracion_fallas_anuales_extremadamente_baja = 0.0095

# Calculo de los pesos combinados para cada tipo de densidad
lambda_ = 0.5  # Factor de ponderación entre 0 y 1

peso_combinado_alta = lambda_ * proporcion_energia_alta + (1 - lambda_) * duracion_fallas_anuales_alta
peso_combinado_media = lambda_ * proporcion_energia_media + (1 - lambda_) * duracion_fallas_anuales_media
peso_combinado_baja = lambda_ * proporcion_energia_baja + (1 - lambda_) * duracion_fallas_anuales_baja
peso_combinado_muy_baja = lambda_ * proporcion_energia_muy_baja + (1 - lambda_) * duracion_fallas_anuales_muy_baja
peso_combinado_extremadamente_baja = lambda_ * proporcion_energia_extremadamente_baja + (1 - lambda_) * duracion_fallas_anuales_extremadamente_baja

# Cálculo de la ENS para cada tipo de densidad
ens_alta = peso_combinado_alta * monto_compensado_anual / (2 * costo_de_falla_2024)
ens_media = peso_combinado_media * monto_compensado_anual / (2 * costo_de_falla_2024)
ens_baja = peso_combinado_baja * monto_compensado_anual / (2 * costo_de_falla_2024)
ens_muy_baja = peso_combinado_muy_baja * monto_compensado_anual / (2 * costo_de_falla_2024)
ens_extremadamente_baja = peso_combinado_extremadamente_baja * monto_compensado_anual / (2 * costo_de_falla_2024)

costo_de_falla_2025 = 467.19			# USD/MWh

for periodo in range(len(lista_dfs_implementacion)):

	# Energía no suministrada eliminada por año (MWh/año)
	ens_eliminada_alta = (ens_alta * reduccion_duracion_falla_alta)
	ens_eliminada_media = (ens_media * reduccion_duracion_falla_media)
	ens_eliminada_baja = (ens_baja * reduccion_duracion_falla_baja)
	ens_eliminada_muy_baja = (ens_muy_baja * reduccion_duracion_falla_muy_baja)
	ens_eliminada_ext_baja = (ens_extremadamente_baja * reduccion_duracion_falla_ext_baja)

	beneficio_compensacion_evitada_alta = ens_eliminada_alta * costo_de_falla_2025 * 2			# USD/año
	beneficio_compensacion_evitada_media = ens_eliminada_media * costo_de_falla_2025 * 2		# USD/año
	beneficio_compensacion_evitada_baja = ens_eliminada_baja * costo_de_falla_2025 * 2			# USD/año
	beneficio_compensacion_evitada_muy_baja = ens_eliminada_muy_baja * costo_de_falla_2025 * 2	# USD/año
	beneficio_compensacion_evitada_ext_baja = ens_eliminada_ext_baja * costo_de_falla_2025 * 2	# USD/año

	# Se asigna una parte a cada segmento según su participación en las compensaciones totales
	cantidad_alta = (df_implementacion["Densidad"] == "ALTA").sum()
	cantidad_media = (df_implementacion["Densidad"] == "MEDIA").sum()
	cantidad_baja = (df_implementacion["Densidad"] == "BAJA").sum()
	cantidad_muy_baja = (df_implementacion["Densidad"] == "MUY BAJA").sum()
	cantidad_extremadamente_baja = (df_implementacion["Densidad"] == "EXTREMADAMENTE BAJA").sum()

	ahorro_compensacion_evitada_parcializado_alto = np.where(df_implementacion["Densidad"] == "ALTA", (beneficio_compensacion_evitada_alta / ((cantidad_alta / 5) * 3)), 0)
	ahorro_compensacion_evitada_parcializado_media = np.where(df_implementacion["Densidad"] == "MEDIA", (beneficio_compensacion_evitada_media / ((cantidad_media / 5) * 3)), 0)
	ahorro_compensacion_evitada_parcializado_bajo = np.where(df_implementacion["Densidad"] == "BAJA", (beneficio_compensacion_evitada_baja / ((cantidad_baja / 5) * 3)), 0)
	ahorro_compensacion_evitada_parcializado_muy_bajo = np.where(df_implementacion["Densidad"] == "MUY BAJA", (beneficio_compensacion_evitada_muy_baja / ((cantidad_muy_baja / 5) * 3)), 0)
	ahorro_compensacion_evitada_parcializado_extremadamente_bajo = np.where(df_implementacion["Densidad"] == "EXTREMADAMENTE BAJA", (beneficio_compensacion_evitada_ext_baja / ((cantidad_extremadamente_baja / 5) * 3)), 0)

	lista_dfs_implementacion[periodo]["Ahorro por compensación evitada [USD/año]"] = (np.where(df_implementacion["Segmento"].isin(["Residencial", "No Residencial BT", "No Residencial AT"]),
																								ahorro_compensacion_evitada_parcializado_alto, 0)
																					+ np.where(df_implementacion["Segmento"].isin(["Residencial", "No Residencial BT", "No Residencial AT"]),
																								ahorro_compensacion_evitada_parcializado_media, 0)
																					+ np.where(df_implementacion["Segmento"].isin(["Residencial", "No Residencial BT", "No Residencial AT"]),
																								ahorro_compensacion_evitada_parcializado_bajo, 0)
																					+ np.where(df_implementacion["Segmento"].isin(["Residencial", "No Residencial BT", "No Residencial AT"]),
																								ahorro_compensacion_evitada_parcializado_muy_bajo, 0)
																					+ np.where(df_implementacion["Segmento"].isin(["Residencial", "No Residencial BT", "No Residencial AT"]),
																								ahorro_compensacion_evitada_parcializado_extremadamente_bajo, 0))


### Beneficio 4) Ahorros por efecto de la reducción del uso de la atención al cliente de las empresas distribuidoras

# Costo promedio de atención al cliente (Fuente: VAD 2024-2028)
atencion_al_cliente = 2.0767			# USD/cliente-año

for periodo in range(len(lista_dfs_implementacion)):

	# Indice para aplicar el cálculo solo al segmento Regulado (Residencial y No_Residencial)
	indice_segmento = np.where(lista_dfs_implementacion[periodo]["Segmento"] == "Residencial", 1,
						np.where(lista_dfs_implementacion[periodo]["Segmento"] == "No Residencial BT", 1,
							   np.where(lista_dfs_implementacion[periodo]["Segmento"] == "No Residencial AT", 1, 0)))

	lista_dfs_implementacion[periodo]["Ahorro atención al cliente [USD/año]"] = (lista_dfs_implementacion[periodo]["Cantidad de clientes"]
																					* atencion_al_cliente
																					* reduccion_atencion_clientes
																					* indice_segmento)
	

### Beneficio 5) Ahorros sobre la inversión evitada en infraestructura de red

# Costo anualizado nacional de inversión en infraestructura de red (Fuente: VAD 2024-2028)
inversion_infraestructura = [199186195, 199332899, 199483685, 201682183, 202155179, 202463045, 
							202684700, 202901013, 204578093, 205011474, 205354661, 205698677, 206106608, 
							206713469, 207322117, 207932556]	 # USD/año

for periodo in range(len(lista_dfs_implementacion)):

	# Inversión evitada
	inversion_evitada = inversion_infraestructura[periodo] * reduccion_infraestructura

	# Se asigna una parte a cada comuna según su "participación" en la inversión total
	ahorro_por_inversion_evitada_parcializada = inversion_evitada / (len(lista_dfs_implementacion[periodo]) / 5)
	lista_dfs_implementacion[periodo]["Ahorro por inversión evitada [USD/año]"] = np.where(lista_dfs_implementacion[periodo]["Segmento"].isin(["Red"]),
																							ahorro_por_inversion_evitada_parcializada, 0)


### Beneficio 6) Se calcula el ahorro anual por reducción del hurto de energía y pérdidas comerciales
	
# Se fija el crecimiento anual compuesto de la demanda (Datos del proceso VAD 2024-2028, 'Beneficios.xlsx')
crecimiento_anual_pt = 0.0000			# % anual de crecimiento de las pérdidas técnicas (0%)
crecimiento_anual_ph = 0.0000			# % anual de crecimiento del hurto de energía (6.43%)
crecimiento_anual_pc = 0.0000			# % anual de crecimiento de las pérdidas comerciales (0%)

for periodo in range(len(lista_dfs_implementacion)):

	lista_dfs_implementacion[periodo]["Pérdidas de energía [pu]"] = (lista_dfs_implementacion[0]["Pérdidas de energía [pu]"]
																		* (1 + crecimiento_anual_pt) ** periodo)
	lista_dfs_implementacion[periodo]["Hurto de energía [pu]"] = (lista_dfs_implementacion[0]["Hurto de energía [pu]"]
																	* (1 + crecimiento_anual_ph) ** periodo)
	lista_dfs_implementacion[periodo]["Pérdidas comerciales [pu]"] = (lista_dfs_implementacion[0]["Pérdidas comerciales [pu]"]
																		* (1 + crecimiento_anual_pc) ** periodo)

	# Perdidas técnicas (pt), pérdidas por hurto (ph) y pérdidas comerciales (pc) nominales
	pt = lista_dfs_implementacion[periodo]["Pérdidas de energía [pu]"]
	ph = lista_dfs_implementacion[periodo]["Hurto de energía [pu]"]
	pc = lista_dfs_implementacion[periodo]["Pérdidas comerciales [pu]"]

	# Cálculo de las pérdidas reducidas
	hurto_reducido = lista_dfs_implementacion[periodo]["Hurto de energía [pu]"] * reduccion_pnt_hurto
	comercial_reducido = lista_dfs_implementacion[periodo]["Pérdidas comerciales [pu]"] * reduccion_pnt_comercial

	# Se calcula de esa forma dado que los % están sobre la energía comprada, no la facturada
	lista_dfs_implementacion[periodo]["Ahorro reducción de PNT [USD/año]"] = ((((lista_dfs_implementacion[periodo]["Energía facturada anual [MWh/año]"] / (1 - pt - ph - pc))
																					* hurto_reducido)
																					* precio_energia_usd_mwh)
																			+ (((lista_dfs_implementacion[periodo]["Energía facturada anual [MWh/año]"] / (1 - pt - ph - pc))
																					* comercial_reducido)
																					* precio_energia_usd_mwh))
	

### Beneficio 7) Gestión del consumo energético residencial mediante tarifas horarias

# Se carga el CMg diferencial 2025 por comuna
cmg_2025 = pd.read_excel("./Referencias de beneficios/Reducción del Precio de la Energía/cmg_diferencial2025_bloque_AC_vs_B.xlsx")
cmg_2025 = cmg_2025.rename(columns={"CMg_diferencial_2025_usd_mwh": "CMg dif 2025 [USD/MWh]"})

prom_nacional = cmg_2025["CMg dif 2025 [USD/MWh]"].mean()

df_imp = df_implementacion.copy()
df_imp = df_imp.reset_index(drop=True)
cmg_map = cmg_2025[['Comuna', 'CMg dif 2025 [USD/MWh]']].copy()
df_imp = df_imp.merge(cmg_map, left_on='Comuna', right_on='Comuna', how='left')

# Se rellenan comunas sin CMg con el promedio nacional
df_imp['CMg dif 2025 [USD/MWh]'] = df_imp['CMg dif 2025 [USD/MWh]'].fillna(prom_nacional)

dif_cmg_2025 = df_imp['CMg dif 2025 [USD/MWh]'].to_numpy()


bloque_caro_de_energia = 0.5126			# % de la energía consumida que se concentra en el bloque caro (horario 18:00 a 7:59hrs)

energia_lavado_de_ropa 	 = 0.016			# % de la energía consumida que representa al lavado de ropa
energia_secado_de_ropa 	 = 0.062			# % de la energía consumida que representa al secado de ropa
energia_plancha_de_ropa  = 0.021			# % de la energía consumida que representa al planchado de ropa
energia_aspiradora 		 = 0.050			# % de la energía consumida que representa al uso de la aspiradora
# energia_hervidora_agua = 0.040			# % de la energía consumida que representa al hervido de agua

clientes_con_lavadora 	= 0.980				# % de los clientes que poseen lavadora
clientes_con_secadora 	= 0.299				# % de los clientes que poseen secadora
clientes_con_plancha 	= 0.693				# % de los clientes que poseen plancha
clientes_con_aspiradora = 0.534				# % de los clientes que poseen aspiradora
# clientes_con_hervidor = 0.779				# % de los clientes que poseen hervidor eléctrico

energia_trasladada_lavado_de_ropa = energia_lavado_de_ropa * clientes_con_lavadora			# % de la energía trasladada del bloque caro al bloque barato sobre el lavado de ropa
energia_trasladada_secado_de_ropa = energia_secado_de_ropa * clientes_con_secadora			# % de la energía trasladada del bloque caro al bloque barato sobre el secado de ropa
energia_trasladada_plancha_de_ropa = energia_plancha_de_ropa * clientes_con_plancha			# % de la energía trasladada del bloque caro al bloque barato sobre el planchado de ropa
energia_trasladada_aspiradora = energia_aspiradora * clientes_con_aspiradora				# % de la energía trasladada del bloque caro al bloque barato sobre el uso de la aspiradora
# energia_trasladada_hervidora_agua = energia_hervidora_agua * clientes_con_hervidor		# % de la energía trasladada del bloque caro al bloque barato sobre el hervido de agua

# Electrificacion extra que se necesitaría considerar para hacer restable el proyecto
# electrificacion = 0.085					# 9,6% de electrificación extra a considerar

electrificacion_alta = 0.120			# mínimo 8% de electrificación extra a considerar
electrificacion_media = 0.055			# mínimo 3% de electrificación extra a considerar
electrificacion_baja = 0.000			# mínimo 0% de electrificación extra a considerar
electrificacion_muy_baja = 0.120		# mínimo 7,5% de electrificación extra a considerar
electrificacion_ext_baja = 0.330		# mínimo 22% de electrificación extra a considerar

electrificacion = np.where(df_implementacion["Densidad"] == "ALTA", electrificacion_alta,
					np.where(df_implementacion["Densidad"] == "MEDIA", electrificacion_media,
						np.where(df_implementacion["Densidad"] == "BAJA", electrificacion_baja,
							np.where(df_implementacion["Densidad"] == "MUY BAJA", electrificacion_muy_baja,
								np.where(df_implementacion["Densidad"] == "EXTREMADAMENTE BAJA", electrificacion_ext_baja, 0)))))


for periodo in range(len(lista_dfs_implementacion)):
	
	# Indice para aplicar el cálculo solo a los segmentos Residencial y No_Residencial
	indice_segmento = np.where(lista_dfs_implementacion[periodo]["Segmento"] == "Residencial", 1, 0)

	lista_dfs_implementacion[periodo]["Ahorro por gestión del consumo [USD/año]"] = (lista_dfs_implementacion[periodo]["Energía facturada anual [MWh/año]"]
																					* indice_segmento
																					* bloque_caro_de_energia
																					* dif_cmg_2025
																					* (energia_trasladada_lavado_de_ropa
																						+ energia_trasladada_secado_de_ropa
																						+ energia_trasladada_plancha_de_ropa
																						+ energia_trasladada_aspiradora))
	
	indice_segmento_regulados = np.where(lista_dfs_implementacion[periodo]["Segmento"] == "Residencial", 1,
										np.where(lista_dfs_implementacion[periodo]["Segmento"] == "No Residencial BT", 1,
											np.where(lista_dfs_implementacion[periodo]["Segmento"] == "No Residencial AT", 1, 0)))
	
	lista_dfs_implementacion[periodo]["Ahorro por gestión del consumo [USD/año]"] += (lista_dfs_implementacion[periodo]["Energía facturada anual [MWh/año]"]
																					* indice_segmento_regulados
																					* bloque_caro_de_energia
																					* dif_cmg_2025
																					* electrificacion)


### Beneficio 7) Ahorro por gestión del consumo energético (según GTD)
# energia_clientes_mayor_consumo = 0.7392
# energia_trasladada = energia_clientes_mayor_consumo * 0.11

# precio_diferencial = 20 * 1.34			# USD/MWh (ajustado por CPI, 2016-2025)

# for periodo in range(len(lista_dfs_implementacion)):
	
# 	# Indice para aplicar el cálculo solo a los segmentos Residencial y No_Residencial
# 	indice_segmento = np.where(lista_dfs_implementacion[periodo]["Segmento"] == "Residencial", 1,
# 						np.where(lista_dfs_implementacion[periodo]["Segmento"] == "No_Residencial", 1, 0))

# 	lista_dfs_implementacion[periodo]["Ahorro por gestión del consumo [USD/año]"] = (lista_dfs_implementacion[periodo]["Energía facturada anual [MWh/año]"]
# 																					* indice_segmento
# 																					* energia_trasladada
# 																					* precio_diferencial)
	

### Beneficio 8) Estimación del ahorro anual por reducción del consumo energético

clientes_afectos_a_reduccion = 0.075		# % de clientes que reducen su consumo energético (7.5%)

for periodo in range(len(lista_dfs_implementacion)):

	clientes_de_densidades_afectas = np.where(lista_dfs_implementacion[periodo]["Densidad"] == "ALTA", 1,
										np.where(lista_dfs_implementacion[periodo]["Densidad"] == "MEDIA", 1,
											np.where(lista_dfs_implementacion[periodo]["Densidad"] == "BAJA", 1, 0)))

	# reducción anual según segmento: Residendial y No Residencial
	reduccion_consumo = np.where(lista_dfs_implementacion[periodo]["Segmento"] == "Residencial", reduccion_consumo_residencial,
							np.where(lista_dfs_implementacion[periodo]["Segmento"] == "No Residencial BT", reduccion_consumo_no_residencial,
								   np.where(lista_dfs_implementacion[periodo]["Segmento"] == "No Residencial AT", reduccion_consumo_no_residencial, 0)))

	# Se calcula Energía facturada anual [MWh/año] * reducción consumo [%] * precio energía [USD/MWh]
	lista_dfs_implementacion[periodo]["Ahorro reducción del consumo [USD/año]"] = (lista_dfs_implementacion[periodo]["Energía facturada anual [MWh/año]"]
																					* clientes_de_densidades_afectas
																					* clientes_afectos_a_reduccion
																					* reduccion_consumo 
																					* precio_energia_usd_mwh)
	

### Beneficio 9) Ahorro nacional anual por reducción de emisiones de CO2 (Reducción de consumo y reducción de pérdidas)

# Toneladas de CO2 emitidas por MWh generado (Fuente: Estudio Ministerio del Medio Ambiente, Ver 'Proyecciones_y_Estimaciones.xlsx')
emisiones_co2_por_mwh = [0.2190, 0.2084, 0.1982, 0.1885, 0.1793, 0.1706, 0.1622, 0.1543,
						 0.1468, 0.1396, 0.1328, 0.1263, 0.1202, 0.1143, 0.1087, 0.1034]

# Estimaciones a partir del informe "Estimación del precio social del carbono para la evaluación de la inversión pública en Chile" (Ver 'Proyecciones_y_Estimaciones.xlsx')
costo_social_co2 = [71.11, 78.83, 86.55, 94.27, 102.00, 109.72, 117.44, 125.16, 
					   132.88, 140.60, 148.33, 156.05, 163.77, 171.49, 179.21, 186.93]		# Costo social USD/tCO2 (Fuente: Ministerio de Desarrollo Social) desde 2025 a 2040

for periodo in range(len(lista_dfs_implementacion)):

	# reducción consumo anual según segmento: Residendial y No_Residencial
	reduccion_consumo = np.where(lista_dfs_implementacion[periodo]["Segmento"] == "Residencial", reduccion_consumo_residencial,
							np.where(lista_dfs_implementacion[periodo]["Segmento"] == "No Residencial BT", reduccion_consumo_no_residencial,
								   np.where(lista_dfs_implementacion[periodo]["Segmento"] == "No Residencial AT", reduccion_consumo_no_residencial, 0)))

	# Perdidas técnicas (pt), pérdidas por hurto (ph) y pérdidas comerciales (pc) nominales
	pt = lista_dfs_implementacion[periodo]["Pérdidas de energía [pu]"]
	ph = lista_dfs_implementacion[periodo]["Hurto de energía [pu]"]
	pc = lista_dfs_implementacion[periodo]["Pérdidas comerciales [pu]"]

	# Cálculo de las pérdidas reducidas
	hurto_reducido = lista_dfs_implementacion[periodo]["Hurto de energía [pu]"] * reduccion_pnt_hurto
	comercial_reducido = lista_dfs_implementacion[periodo]["Pérdidas comerciales [pu]"] * reduccion_pnt_comercial
	
	lista_dfs_implementacion[periodo]["Ahorro reducción emisiones CO2 [USD/año]"] = ((lista_dfs_implementacion[periodo]["Energía facturada anual [MWh/año]"]
																				   		* clientes_de_densidades_afectas
																						* clientes_afectos_a_reduccion
																						* reduccion_consumo
																						* emisiones_co2_por_mwh[periodo]
																						* costo_social_co2[periodo])
																					+ ((lista_dfs_implementacion[periodo]["Energía facturada anual [MWh/año]"] / (1 - pt - ph - pc))
																						* hurto_reducido
																						* emisiones_co2_por_mwh[periodo]
																						* costo_social_co2[periodo]))
																					# + (((lista_dfs_implementacion[periodo]["Energía facturada anual [MWh/año]"] / (1 - pt - ph - pc))
																					# 	* comercial_reducido
																					# 	* emisiones_co2_por_mwh[periodo]
																					# 	* costo_social_co2[periodo])))


### Suma de todos los beneficios anuales y creación del DataFrame resumen de beneficios
df_resultado_final_beneficios = pd.DataFrame()
df_resultado_final_beneficios_alta = pd.DataFrame()
df_resultado_final_beneficios_media = pd.DataFrame()
df_resultado_final_beneficios_baja = pd.DataFrame()
df_resultado_final_beneficios_muy_baja = pd.DataFrame()
df_resultado_final_beneficios_ext_baja = pd.DataFrame()

beneficios = [
	"Ahorro en lectura pedestre [USD/año]",
	"Ahorro corte y reposición remoto [USD/año]",
	"Ahorro reducción de PNT [USD/año]",
	"Ahorro por compensación evitada [USD/año]",
	"Ahorro por gestión del consumo [USD/año]",
	"Ahorro reducción del consumo [USD/año]",
	"Ahorro reducción emisiones CO2 [USD/año]",
	"Ahorro atención al cliente [USD/año]",
	"Ahorro por inversión evitada [USD/año]"
]

df_resultado_final_beneficios["Concepto"] = beneficios
df_resultado_final_beneficios_alta["Concepto"] = beneficios
df_resultado_final_beneficios_media["Concepto"] = beneficios
df_resultado_final_beneficios_baja["Concepto"] = beneficios
df_resultado_final_beneficios_muy_baja["Concepto"] = beneficios
df_resultado_final_beneficios_ext_baja["Concepto"] = beneficios

for periodo in range(len(lista_dfs_implementacion)):

	df_final = lista_dfs_implementacion[periodo]
	df_alta = lista_dfs_implementacion[periodo][lista_dfs_implementacion[periodo]["Densidad"] == "ALTA"]
	df_media = lista_dfs_implementacion[periodo][lista_dfs_implementacion[periodo]["Densidad"] == "MEDIA"]
	df_baja = lista_dfs_implementacion[periodo][lista_dfs_implementacion[periodo]["Densidad"] == "BAJA"]
	df_muy_baja = lista_dfs_implementacion[periodo][lista_dfs_implementacion[periodo]["Densidad"] == "MUY BAJA"]
	df_ext_baja = lista_dfs_implementacion[periodo][lista_dfs_implementacion[periodo]["Densidad"] == "EXTREMADAMENTE BAJA"]

	nuevo_periodo_beneficios = [
		df_final["Ahorro en lectura pedestre [USD/año]"].sum(),
		df_final["Ahorro corte y reposición remoto [USD/año]"].sum(),
		df_final["Ahorro reducción de PNT [USD/año]"].sum(),
		df_final["Ahorro por compensación evitada [USD/año]"].sum(),
		df_final["Ahorro por gestión del consumo [USD/año]"].sum(),
		df_final["Ahorro reducción del consumo [USD/año]"].sum(),
		df_final["Ahorro reducción emisiones CO2 [USD/año]"].sum(),
		df_final["Ahorro atención al cliente [USD/año]"].sum(),
		df_final["Ahorro por inversión evitada [USD/año]"].sum()
	]

	nuevo_periodo_beneficios_alta = [
		df_alta["Ahorro en lectura pedestre [USD/año]"].sum(),
		df_alta["Ahorro corte y reposición remoto [USD/año]"].sum(),
		df_alta["Ahorro reducción de PNT [USD/año]"].sum(),
		df_alta["Ahorro por compensación evitada [USD/año]"].sum(),
		df_alta["Ahorro por gestión del consumo [USD/año]"].sum(),
		df_alta["Ahorro reducción del consumo [USD/año]"].sum(),
		df_alta["Ahorro reducción emisiones CO2 [USD/año]"].sum(),
		df_alta["Ahorro atención al cliente [USD/año]"].sum(),
		df_alta["Ahorro por inversión evitada [USD/año]"].sum()
	]

	nuevo_periodo_beneficios_media = [
		df_media["Ahorro en lectura pedestre [USD/año]"].sum(),
		df_media["Ahorro corte y reposición remoto [USD/año]"].sum(),
		df_media["Ahorro reducción de PNT [USD/año]"].sum(),
		df_media["Ahorro por compensación evitada [USD/año]"].sum(),
		df_media["Ahorro por gestión del consumo [USD/año]"].sum(),
		df_media["Ahorro reducción del consumo [USD/año]"].sum(),
		df_media["Ahorro reducción emisiones CO2 [USD/año]"].sum(),
		df_media["Ahorro atención al cliente [USD/año]"].sum(),
		df_media["Ahorro por inversión evitada [USD/año]"].sum()
	]

	nuevo_periodo_beneficios_baja = [
		df_baja["Ahorro en lectura pedestre [USD/año]"].sum(),
		df_baja["Ahorro corte y reposición remoto [USD/año]"].sum(),
		df_baja["Ahorro reducción de PNT [USD/año]"].sum(),
		df_baja["Ahorro por compensación evitada [USD/año]"].sum(),
		df_baja["Ahorro por gestión del consumo [USD/año]"].sum(),
		df_baja["Ahorro reducción del consumo [USD/año]"].sum(),
		df_baja["Ahorro reducción emisiones CO2 [USD/año]"].sum(),
		df_baja["Ahorro atención al cliente [USD/año]"].sum(),
		df_baja["Ahorro por inversión evitada [USD/año]"].sum()
	]

	nuevo_periodo_beneficios_muy_baja = [
		df_muy_baja["Ahorro en lectura pedestre [USD/año]"].sum(),
		df_muy_baja["Ahorro corte y reposición remoto [USD/año]"].sum(),
		df_muy_baja["Ahorro reducción de PNT [USD/año]"].sum(),
		df_muy_baja["Ahorro por compensación evitada [USD/año]"].sum(),
		df_muy_baja["Ahorro por gestión del consumo [USD/año]"].sum(),
		df_muy_baja["Ahorro reducción del consumo [USD/año]"].sum(),
		df_muy_baja["Ahorro reducción emisiones CO2 [USD/año]"].sum(),
		df_muy_baja["Ahorro atención al cliente [USD/año]"].sum(),
		df_muy_baja["Ahorro por inversión evitada [USD/año]"].sum()
	]

	nuevo_periodo_beneficios_ext_baja = [
		df_ext_baja["Ahorro en lectura pedestre [USD/año]"].sum(),
		df_ext_baja["Ahorro corte y reposición remoto [USD/año]"].sum(),
		df_ext_baja["Ahorro reducción de PNT [USD/año]"].sum(),
		df_ext_baja["Ahorro por compensación evitada [USD/año]"].sum(),
		df_ext_baja["Ahorro por gestión del consumo [USD/año]"].sum(),
		df_ext_baja["Ahorro reducción del consumo [USD/año]"].sum(),
		df_ext_baja["Ahorro reducción emisiones CO2 [USD/año]"].sum(),
		df_ext_baja["Ahorro atención al cliente [USD/año]"].sum(),
		df_ext_baja["Ahorro por inversión evitada [USD/año]"].sum()
	]

	df_resultado_final_beneficios["Periodo {}".format(periodo)] = nuevo_periodo_beneficios
	df_resultado_final_beneficios_alta["Periodo {}".format(periodo)] = nuevo_periodo_beneficios_alta
	df_resultado_final_beneficios_media["Periodo {}".format(periodo)] = nuevo_periodo_beneficios_media
	df_resultado_final_beneficios_baja["Periodo {}".format(periodo)] = nuevo_periodo_beneficios_baja
	df_resultado_final_beneficios_muy_baja["Periodo {}".format(periodo)] = nuevo_periodo_beneficios_muy_baja
	df_resultado_final_beneficios_ext_baja["Periodo {}".format(periodo)] = nuevo_periodo_beneficios_ext_baja

	# Suma de todos los beneficios anuales
	lista_dfs_implementacion[periodo]["Beneficio total anual [USD/año]"] = (lista_dfs_implementacion[periodo]["Ahorro en lectura pedestre [USD/año]"]				# Beneficio 1
																			+ lista_dfs_implementacion[periodo]["Ahorro corte y reposición remoto [USD/año]"]		# Beneficio 2
																			+ lista_dfs_implementacion[periodo]["Ahorro reducción de PNT [USD/año]"]				# Beneficio 3
																			+ lista_dfs_implementacion[periodo]["Ahorro por compensación evitada [USD/año]"]		# Beneficio 4
																			+ lista_dfs_implementacion[periodo]["Ahorro por gestión del consumo [USD/año]"]			# Beneficio 5
																			+ lista_dfs_implementacion[periodo]["Ahorro reducción del consumo [USD/año]"]			# Beneficio 6
																			+ lista_dfs_implementacion[periodo]["Ahorro reducción emisiones CO2 [USD/año]"]			# Beneficio 7
																			+ lista_dfs_implementacion[periodo]["Ahorro atención al cliente [USD/año]"]				# Beneficio 8
																			+ lista_dfs_implementacion[periodo]["Ahorro por inversión evitada [USD/año]"])			# Beneficio 9
	
	print(" -> Periodo {}) Beneficio total anual del proyecto: M USD {:,.4f}"
			  .format(periodo, lista_dfs_implementacion[periodo]["Beneficio total anual [USD/año]"].sum() / 1_000_000))
	
df_resultado_final = pd.concat([df_resultado_final, df_resultado_final_beneficios], ignore_index=True)
df_resultado_final_alta = pd.concat([df_resultado_final_alta, df_resultado_final_beneficios_alta], ignore_index=True)
df_resultado_final_media = pd.concat([df_resultado_final_media, df_resultado_final_beneficios_media], ignore_index=True)
df_resultado_final_baja = pd.concat([df_resultado_final_baja, df_resultado_final_beneficios_baja], ignore_index=True)
df_resultado_final_muy_baja = pd.concat([df_resultado_final_muy_baja, df_resultado_final_beneficios_muy_baja], ignore_index=True)
df_resultado_final_ext_baja = pd.concat([df_resultado_final_ext_baja, df_resultado_final_beneficios_ext_baja], ignore_index=True)


### Exportación de resultados a un archivo Excel
# Se encuentra la tecnología que se está evaluando
tecnologia = df_implementacion["Tecnología"].unique()

if len(tecnologia) > 1:
	nombre_archivo_global = "./Escenario Completo (Esperable)/Exigencia SMI/Resultado_de_Implementacion_SMI_Mixto.xlsx"
	nombre_archivo_alta = "./Escenario Completo (Esperable)/Exigencia SMI/Resultado_de_Implementacion_SMI_Mixto (ALTA).xlsx"
	nombre_archivo_media = "./Escenario Completo (Esperable)/Exigencia SMI/Resultado_de_Implementacion_SMI_Mixto (MEDIA).xlsx"
	nombre_archivo_baja = "./Escenario Completo (Esperable)/Exigencia SMI/Resultado_de_Implementacion_SMI_Mixto (BAJA).xlsx"
	nombre_archivo_muy_baja = "./Escenario Completo (Esperable)/Exigencia SMI/Resultado_de_Implementacion_SMI_Mixto (MUY BAJA).xlsx"
	nombre_archivo_ext_baja = "./Escenario Completo (Esperable)/Exigencia SMI/Resultado_de_Implementacion_SMI_Mixto (EXT BAJA).xlsx"

	# Se guarda el DataFrame en un archivo Excel nuevo
	df_implementacion.to_excel(nombre_archivo_global, sheet_name="Implementacion_SMI_Mixto", index=False)
	df_implementacion.to_excel(nombre_archivo_alta, sheet_name="Implementacion_SMI_Mixto", index=False)
	df_implementacion.to_excel(nombre_archivo_media, sheet_name="Implementacion_SMI_Mixto", index=False)
	df_implementacion.to_excel(nombre_archivo_baja, sheet_name="Implementacion_SMI_Mixto", index=False)
	df_implementacion.to_excel(nombre_archivo_muy_baja, sheet_name="Implementacion_SMI_Mixto", index=False)
	df_implementacion.to_excel(nombre_archivo_ext_baja, sheet_name="Implementacion_SMI_Mixto", index=False)

	with pd.ExcelWriter(
	nombre_archivo_global,
	engine="openpyxl",
	mode="a",                	# append (agregar)
	if_sheet_exists="replace"   # o "overlay" / "new" según tu versión
	) as writer:
		df_resultado_final.to_excel(writer, sheet_name="CB_SMI_Mixto", index=False)

	with pd.ExcelWriter(
	nombre_archivo_alta,
	engine="openpyxl",
	mode="a",                	# append (agregar)
	if_sheet_exists="replace"   # o "overlay" / "new" según tu versión
	) as writer:
		df_resultado_final_alta.to_excel(writer, sheet_name="CB_SMI_Mixto_ALTA", index=False)

	with pd.ExcelWriter(
	nombre_archivo_media,
	engine="openpyxl",
	mode="a",                	# append (agregar)
	if_sheet_exists="replace"   # o "overlay" / "new" según tu versión
	) as writer:
		df_resultado_final_media.to_excel(writer, sheet_name="CB_SMI_Mixto_MEDIA", index=False)

	with pd.ExcelWriter(
	nombre_archivo_baja,
	engine="openpyxl",
	mode="a",                	# append (agregar)
	if_sheet_exists="replace"   # o "overlay" / "new" según tu versión
	) as writer:
		df_resultado_final_baja.to_excel(writer, sheet_name="CB_SMI_Mixto_BAJA", index=False)

	with pd.ExcelWriter(
	nombre_archivo_muy_baja,
	engine="openpyxl",
	mode="a",                	# append (agregar)
	if_sheet_exists="replace"   # o "overlay" / "new" según tu versión
	) as writer:
		df_resultado_final_muy_baja.to_excel(writer, sheet_name="CB_SMI_Mixto_MUY_BAJA", index=False)

	with pd.ExcelWriter(
	nombre_archivo_ext_baja,
	engine="openpyxl",
	mode="a",                	# append (agregar)
	if_sheet_exists="replace"   # o "overlay" / "new" según tu versión
	) as writer:
		df_resultado_final_ext_baja.to_excel(writer, sheet_name="CB_SMI_Mixto_EXT_BAJA", index=False)	

else:
	nombre_archivo_global = f"./Escenario Completo (Esperable)/Exigencia SMI/Resultado_de_Implementacion_SMI_{tecnologia[0]}.xlsx"
	nombre_archivo_alta = f"./Escenario Completo (Esperable)/Exigencia SMI/Resultado_de_Implementacion_SMI_{tecnologia[0]}.xlsx"
	nombre_archivo_media = f"./Escenario Completo (Esperable)/Exigencia SMI/Resultado_de_Implementacion_SMI_{tecnologia[0]}.xlsx"
	nombre_archivo_baja = f"./Escenario Completo (Esperable)/Exigencia SMI/Resultado_de_Implementacion_SMI_{tecnologia[0]}.xlsx"
	nombre_archivo_muy_baja = f"./Escenario Completo (Esperable)/Exigencia SMI/Resultado_de_Implementacion_SMI_{tecnologia[0]}.xlsx"
	nombre_archivo_ext_baja = f"./Escenario Completo (Esperable)/Exigencia SMI/Resultado_de_Implementacion_SMI_{tecnologia[0]}.xlsx"

	# Se guarda el DataFrame en un archivo Excel nuevo
	df_implementacion.to_excel(nombre_archivo_global, sheet_name=f"Implementacion_SMI_{tecnologia[0]}", index=False)
	df_implementacion.to_excel(nombre_archivo_alta, sheet_name=f"Implementacion_SMI_{tecnologia[0]}", index=False)
	df_implementacion.to_excel(nombre_archivo_media, sheet_name=f"Implementacion_SMI_{tecnologia[0]}", index=False)
	df_implementacion.to_excel(nombre_archivo_baja, sheet_name=f"Implementacion_SMI_{tecnologia[0]}", index=False)
	df_implementacion.to_excel(nombre_archivo_muy_baja, sheet_name=f"Implementacion_SMI_{tecnologia[0]}", index=False)
	df_implementacion.to_excel(nombre_archivo_ext_baja, sheet_name=f"Implementacion_SMI_{tecnologia[0]}", index=False)

	with pd.ExcelWriter(
	nombre_archivo_global,
	engine="openpyxl",
	mode="a",                	# append (agregar)
	if_sheet_exists="replace"   # o "overlay" / "new" según tu versión
	) as writer:
		df_resultado_final_alta.to_excel(writer, sheet_name=f"CB_SMI_{tecnologia[0]}", index=False)

	with pd.ExcelWriter(
	nombre_archivo_alta,
	engine="openpyxl",
	mode="a",                	# append (agregar)
	if_sheet_exists="replace"   # o "overlay" / "new" según tu versión
	) as writer:
		df_resultado_final_alta.to_excel(writer, sheet_name=f"CB_SMI_{tecnologia[0]}_ALTA", index=False)

	with pd.ExcelWriter(
	nombre_archivo_media,
	engine="openpyxl",
	mode="a",                	# append (agregar)
	if_sheet_exists="replace"   # o "overlay" / "new" según tu versión
	) as writer:
		df_resultado_final_media.to_excel(writer, sheet_name=f"CB_SMI_{tecnologia[0]}_MEDIA", index=False)

	with pd.ExcelWriter(
	nombre_archivo_baja,
	engine="openpyxl",
	mode="a",                	# append (agregar)
	if_sheet_exists="replace"   # o "overlay" / "new" según tu versión
	) as writer:
		df_resultado_final_baja.to_excel(writer, sheet_name=f"CB_SMI_{tecnologia[0]}_BAJA", index=False)

	with pd.ExcelWriter(
	nombre_archivo_muy_baja,
	engine="openpyxl",
	mode="a",                	# append (agregar)
	if_sheet_exists="replace"   # o "overlay" / "new" según tu versión
	) as writer:
		df_resultado_final_muy_baja.to_excel(writer, sheet_name=f"CB_SMI_{tecnologia[0]}_MUY_BAJA", index=False)

	with pd.ExcelWriter(
	nombre_archivo_ext_baja,
	engine="openpyxl",
	mode="a",                	# append (agregar)
	if_sheet_exists="replace"   # o "overlay" / "new" según tu versión
	) as writer:
		df_resultado_final_ext_baja.to_excel(writer, sheet_name=f"CB_SMI_{tecnologia[0]}_EXT_BAJA", index=False)

 -> Periodo 0) Beneficio total anual del proyecto: M USD 278.7365
 -> Periodo 1) Beneficio total anual del proyecto: M USD 286.8574
 -> Periodo 2) Beneficio total anual del proyecto: M USD 295.1081
 -> Periodo 3) Beneficio total anual del proyecto: M USD 303.6579
 -> Periodo 4) Beneficio total anual del proyecto: M USD 312.2360
 -> Periodo 5) Beneficio total anual del proyecto: M USD 320.9675
 -> Periodo 6) Beneficio total anual del proyecto: M USD 329.8460
 -> Periodo 7) Beneficio total anual del proyecto: M USD 338.9132
 -> Periodo 8) Beneficio total anual del proyecto: M USD 348.2788
 -> Periodo 9) Beneficio total anual del proyecto: M USD 357.7343
 -> Periodo 10) Beneficio total anual del proyecto: M USD 367.3960
 -> Periodo 11) Beneficio total anual del proyecto: M USD 377.2613
 -> Periodo 12) Beneficio total anual del proyecto: M USD 387.3704
 -> Periodo 13) Beneficio total anual del proyecto: M USD 397.7043
 -> Periodo 14) Beneficio total anual del proyecto: M USD 408.2796
 -> P

In [57]:
### Capacidad mensual según tecnología de comunicaciones

# Se asignan los resultados del último período (requerimientos máximos) para análisis requerimientos de datos
df_requerimientos = df_implementacion_15.copy()

# Se rellenan los valores 0 de MB Mensuales por DCU con 1 (¡Esto es sólo para considerar los datos de todas las comunas, no tiene impacto en el análisis económico)
df_requerimientos.loc[((df_requerimientos["Cantidad DCU"] == 0) & (df_requerimientos["Segmento"] == "Red")), "Cantidad DCU"] = 1

# Se calcula la cantidad de datos transmitida por DCU (Cálulos realizados en 'Costos.xlsx')

# 1: Se consideran las exigencias de medición de la Norma SMMC hasta antes del 2034 (Artículo 9-8)
# 0: Se consideran las exigencias de medición de la Norma SMMC desde el 2034 en adelante (Artículo 9-8)
# 2: Se consideran las exigencias de medición del SMI.
exigencias_de_medicion = 1		# Input: 1 o 0

if exigencias_de_medicion == 1:
	datos_cliente_3f_zona_alta = 110.2 / 1024		# kB mensual por cliente
	datos_cliente_3f_zona_baja = 45.7 / 1024		# kB mensual por cliente
	datos_cliente_1f_zona_alta = 71.7 / 1024		# kB mensual por cliente
	datos_cliente_1f_zona_baja = 32.5 / 1024		# kB mensual por cliente

	datos_trafo_3f_zona_alta = 110.2 / 1024		# kB mensual por trafo
	datos_trafo_3f_zona_baja = 45.7 / 1024		# kB mensual por trafo
	datos_trafo_1f_zona_alta = 83.7 / 1024		# kB mensual por trafo
	datos_trafo_1f_zona_baja = 38.5 / 1024		# kB mensual por trafo

elif exigencias_de_medicion == 0:
	datos_cliente_3f_zona_alta = 110.2 / 1024		# kB mensual por cliente
	datos_cliente_3f_zona_baja = 110.2 / 1024		# kB mensual por cliente
	datos_cliente_1f_zona_alta = 71.7 / 1024		# kB mensual por cliente
	datos_cliente_1f_zona_baja = 71.7 / 1024		# kB mensual por cliente

	datos_trafo_3f_zona_alta = 110.2 / 1024		# kB mensual por trafo
	datos_trafo_3f_zona_baja = 110.2 / 1024		# kB mensual por trafo
	datos_trafo_1f_zona_alta = 83.7 / 1024		# kB mensual por trafo
	datos_trafo_1f_zona_baja = 83.7 / 1024		# kB mensual por trafo

else:	# exigencias_de_medicion == 2
	datos_cliente_3f_zona_alta = 38.5 / 1024		# kB mensual por cliente
	datos_cliente_3f_zona_baja = 38.5 / 1024		# kB mensual por cliente
	datos_cliente_1f_zona_alta = 32.5 / 1024		# kB mensual por cliente
	datos_cliente_1f_zona_baja = 32.5 / 1024		# kB mensual por cliente

	datos_trafo_3f_zona_alta = 45.7 / 1024		# kB mensual por trafo
	datos_trafo_3f_zona_baja = 45.7 / 1024		# kB mensual por trafo
	datos_trafo_1f_zona_alta = 38.5 / 1024		# kB mensual por trafo
	datos_trafo_1f_zona_baja = 38.5 / 1024		# kB mensual por trafo

valores = []

for i, row in df_requerimientos.iloc[4::5].iterrows():			# Recorrer solo las filas de segmento "Red"		
	cantidad_dcu = df_requerimientos.loc[i, "Cantidad DCU"]

	# Se suman las cantidades correspondientes de UM enfocadas a clientes y transformadores según la densidad de la zona
	if cantidad_dcu > 0 and (row["Densidad"] == "ALTA" or row["Densidad"] == "MEDIA"):
		total_clientes_3f_zona_alta = df_requerimientos.loc[(row["Comuna"], ["No Residencial AT", "LibreDx"]), "Cantidad de clientes"].sum()
		total_clientes_1f_zona_alta = df_requerimientos.loc[(row["Comuna"], ["Residencial", "No Residencial BT"]), "Cantidad de clientes"].sum()

		total_trafo_3f_zona_alta = (df_requerimientos.loc[(row["Comuna"], ["Red"]), "UM 3F 1S para TD"].sum()
									+ df_requerimientos.loc[(row["Comuna"], ["Red"]), "UM 3F 2S para TD"].sum())
		total_trafo_1f_zona_alta = df_requerimientos.loc[(row["Comuna"], ["Red"]), "UM 1F para TD"].sum()

		mb_exigidos = (datos_cliente_3f_zona_alta * total_clientes_3f_zona_alta + datos_cliente_1f_zona_alta * total_clientes_1f_zona_alta
								+ datos_trafo_3f_zona_alta * total_trafo_3f_zona_alta + datos_trafo_1f_zona_alta * total_trafo_1f_zona_alta) / cantidad_dcu
		valores.append(mb_exigidos)

	elif cantidad_dcu > 0 and (row["Densidad"] == "BAJA" or row["Densidad"] == "MUY BAJA" or row["Densidad"] == "EXTREMADAMENTE BAJA"):
		total_clientes_3f_zona_baja = df_requerimientos.loc[(row["Comuna"], ["No Residencial AT", "LibreDx"]), "Cantidad de clientes"].sum()
		total_clientes_1f_zona_baja = df_requerimientos.loc[(row["Comuna"], ["Residencial", "No Residencial BT"]), "Cantidad de clientes"].sum()

		total_trafo_3f_zona_baja = (df_requerimientos.loc[(row["Comuna"], ["Red"]), "UM 3F 1S para TD"].sum()
									+ df_requerimientos.loc[(row["Comuna"], ["Red"]), "UM 3F 2S para TD"].sum())
		total_trafo_1f_zona_baja = df_requerimientos.loc[(row["Comuna"], ["Red"]), "UM 1F para TD"].sum()

		mb_exigidos = (datos_cliente_3f_zona_baja * total_clientes_3f_zona_baja + datos_cliente_1f_zona_baja * total_clientes_1f_zona_baja
								+ datos_trafo_3f_zona_baja * total_trafo_3f_zona_baja + datos_trafo_1f_zona_baja * total_trafo_1f_zona_baja) / cantidad_dcu
		valores.append(mb_exigidos)

	else:        
		mb_exigidos = 0								# Caso de Celular, TWACS y otros segmentos
		valores.append(mb_exigidos)
		
df_requerimientos["MB Mensuales por DCU"] = 0.0
filtro = df_requerimientos["Segmento"] == "Red"

df_requerimientos.loc[filtro, "MB Mensuales por DCU"] = pd.Series(valores, index=df_requerimientos.index[filtro]).round(4)


### Capacidad mensual según tecnología de comunicaciones
capacidad_twacs_por_dcu = ((300 / 8) * 60 * 60 * 24 * 30) / 1_000_000				# 300bps a 97,2MB mensuales por DCU TWACS
capacidad_g3plc_por_dcu = ((100 * 1000 / 8) * 60 * 60 * 24 * 30) / 1_000_000			# 35kbps a 11.340MB mensuales por DCU G3-PLC

capacidad_lora_por_dcu = ((50 * 1000 / 8) * 60 * 60 * 24 * 30) / 1_000_000		# 6,944kbps a 2.249MB mensuales por DCU LoRaWAN
capacidad_rfmesh_por_dcu = ((40 * 1000 / 8) * 60 * 60 * 24 * 30) / 1_000_000		# 40kbps a 12.960MB mensuales por DCU RF-Mesh


# Variables auxiliares para contabilizar las comunas con capacidad suficiente
comunas_con_capacidad_suficiente_alta = 0
comunas_con_capacidad_suficiente_media = 0
comunas_con_capacidad_suficiente_baja = 0
comunas_con_capacidad_suficiente_muy_baja = 0
comunas_con_capacidad_suficiente_extremadamente_baja = 0

# Se recorre la fila "red" de cada comuna
for i, row in df_requerimientos.iloc[4::5].iterrows():			# Recorrer solo las filas de segmento "Red"
	densidad = row["Densidad"]
	mb_mensuales = row["MB Mensuales por DCU"]
	tecnologia = row["Tecnología"]

	if densidad == "ALTA":
		if tecnologia == "TWACS":
			comunas_con_capacidad_suficiente_alta += 1 if mb_mensuales < capacidad_twacs_por_dcu else 0
		if tecnologia == "G3-PLC":
			comunas_con_capacidad_suficiente_alta += 1 if mb_mensuales < capacidad_g3plc_por_dcu else 0	
		if tecnologia == "LoRa":
			comunas_con_capacidad_suficiente_alta += 1 if mb_mensuales < capacidad_lora_por_dcu else 0
		if tecnologia == "RF-Mesh":
			comunas_con_capacidad_suficiente_alta += 1 if mb_mensuales < capacidad_rfmesh_por_dcu else 0	
	
	elif densidad == "MEDIA":
		if tecnologia == "TWACS":
			comunas_con_capacidad_suficiente_media += 1 if mb_mensuales < capacidad_twacs_por_dcu else 0
		if tecnologia == "G3-PLC":
			comunas_con_capacidad_suficiente_media += 1 if mb_mensuales < capacidad_g3plc_por_dcu else 0	
		if tecnologia == "LoRa":
			comunas_con_capacidad_suficiente_media += 1 if mb_mensuales < capacidad_lora_por_dcu else 0
		if tecnologia == "RF-Mesh":
			comunas_con_capacidad_suficiente_media += 1 if mb_mensuales < capacidad_rfmesh_por_dcu else 0

	elif densidad == "BAJA":
		if tecnologia == "TWACS":
			comunas_con_capacidad_suficiente_baja += 1 if mb_mensuales < capacidad_twacs_por_dcu else 0
		if tecnologia == "G3-PLC":
			comunas_con_capacidad_suficiente_baja += 1 if mb_mensuales < capacidad_g3plc_por_dcu else 0	
		if tecnologia == "LoRa":
			comunas_con_capacidad_suficiente_baja += 1 if mb_mensuales < capacidad_lora_por_dcu else 0
		if tecnologia == "RF-Mesh":
			comunas_con_capacidad_suficiente_baja += 1 if mb_mensuales < capacidad_rfmesh_por_dcu else 0

	elif densidad == "MUY BAJA":
		if tecnologia == "TWACS":
			comunas_con_capacidad_suficiente_muy_baja += 1 if mb_mensuales < capacidad_twacs_por_dcu else 0
		if tecnologia == "G3-PLC":
			comunas_con_capacidad_suficiente_muy_baja += 1 if mb_mensuales < capacidad_g3plc_por_dcu else 0	
		if tecnologia == "LoRa":
			comunas_con_capacidad_suficiente_muy_baja += 1 if mb_mensuales < capacidad_lora_por_dcu else 0
		if tecnologia == "RF-Mesh":
			comunas_con_capacidad_suficiente_muy_baja += 1 if mb_mensuales < capacidad_rfmesh_por_dcu else 0

	else:	# EXTREMADAMENTE BAJA
		if tecnologia == "TWACS":
			comunas_con_capacidad_suficiente_extremadamente_baja += 1 if mb_mensuales < capacidad_twacs_por_dcu else 0
		if tecnologia == "G3-PLC":
			comunas_con_capacidad_suficiente_extremadamente_baja += 1 if mb_mensuales < capacidad_g3plc_por_dcu else 0	
		if tecnologia == "LoRa":
			comunas_con_capacidad_suficiente_extremadamente_baja += 1 if mb_mensuales < capacidad_lora_por_dcu else 0
		if tecnologia == "RF-Mesh":
			comunas_con_capacidad_suficiente_extremadamente_baja += 1 if mb_mensuales < capacidad_rfmesh_por_dcu else 0
	
total_comunas_alta = len(df_requerimientos[df_requerimientos['Densidad'] == 'ALTA']) // 5
total_comunas_media = len(df_requerimientos[df_requerimientos['Densidad'] == 'MEDIA']) // 5
total_comunas_baja = len(df_requerimientos[df_requerimientos['Densidad'] == 'BAJA']) // 5
total_comunas_muy_baja = len(df_requerimientos[df_requerimientos['Densidad'] == 'MUY BAJA']) // 5
total_comunas_extremadamente_baja = len(df_requerimientos[df_requerimientos['Densidad'] == 'EXTREMADAMENTE BAJA']) // 5

# Resultados de capacidad suficiente
for densidad, tecnologia in mapeo_tecnologia.items():
	print(f" - {densidad}: {tecnologia}")
print("----------------------------------------------------------------------")
print("Capacidad mensual según tecnología de comunicaciones:")
print(f" - Comunas con densidad ALTA y capacidad suficiente: {comunas_con_capacidad_suficiente_alta} de {total_comunas_alta}, ({(comunas_con_capacidad_suficiente_alta / total_comunas_alta) * 100:.2f}%)")
print(f" - Comunas con densidad MEDIA y capacidad suficiente: {comunas_con_capacidad_suficiente_media} de {total_comunas_media}, ({(comunas_con_capacidad_suficiente_media / total_comunas_media) * 100:.2f}%)")
print(f" - Comunas con densidad BAJA y capacidad suficiente: {comunas_con_capacidad_suficiente_baja} de {total_comunas_baja}, ({(comunas_con_capacidad_suficiente_baja / total_comunas_baja) * 100:.2f}%)")
print(f" - Comunas con densidad MUY BAJA y capacidad suficiente: {comunas_con_capacidad_suficiente_muy_baja} de {total_comunas_muy_baja}, ({(comunas_con_capacidad_suficiente_muy_baja / total_comunas_muy_baja) * 100:.2f}%)")
print(f" - Comunas con densidad EXTREMADAMENTE BAJA y capacidad suficiente: {comunas_con_capacidad_suficiente_extremadamente_baja} de {total_comunas_extremadamente_baja}, ({(comunas_con_capacidad_suficiente_extremadamente_baja / 
																																											 total_comunas_extremadamente_baja) * 100:.2f}%)")

 - EXTREMADAMENTE BAJA: LoRa
 - MUY BAJA: LoRa
 - BAJA: RF-Mesh
 - MEDIA: G3-PLC
 - ALTA: G3-PLC
----------------------------------------------------------------------
Capacidad mensual según tecnología de comunicaciones:
 - Comunas con densidad ALTA y capacidad suficiente: 31 de 31, (100.00%)
 - Comunas con densidad MEDIA y capacidad suficiente: 42 de 42, (100.00%)
 - Comunas con densidad BAJA y capacidad suficiente: 49 de 49, (100.00%)
 - Comunas con densidad MUY BAJA y capacidad suficiente: 101 de 101, (100.00%)
 - Comunas con densidad EXTREMADAMENTE BAJA y capacidad suficiente: 107 de 107, (100.00%)


In [58]:
### Calculo sobre los costos netos de inversión y operación por tipo de densidad
costos_por_densidad = df_implementacion.groupby(["Tecnología", "Densidad"]).agg({
	"Inversión Total [USD]": "sum",
	"Costos Operacionales Totales [USD/año]": "sum"
}).reset_index()

# Se busca la cantidad de clientes por tipo de densidad
total_clientes_alta = df_implementacion[df_implementacion['Densidad'] == 'ALTA']['Cantidad de clientes'].sum()
total_clientes_media = df_implementacion[df_implementacion['Densidad'] == 'MEDIA']['Cantidad de clientes'].sum()
total_clientes_baja = df_implementacion[df_implementacion['Densidad'] == 'BAJA']['Cantidad de clientes'].sum()
total_clientes_muy_baja = df_implementacion[df_implementacion['Densidad'] == 'MUY BAJA']['Cantidad de clientes'].sum()
total_clientes_extremadamente_baja = df_implementacion[df_implementacion['Densidad'] == 'EXTREMADAMENTE BAJA']['Cantidad de clientes'].sum()

for i, row in costos_por_densidad.iterrows():
	densidad = row["Densidad"]
	inversion_total = row["Inversión Total [USD]"]
	costos_operacionales = row["Costos Operacionales Totales [USD/año]"]

	if densidad == "ALTA":
		tecnología_alta = row["Tecnología"]
		inversion_promedio_alta = inversion_total / total_clientes_alta		# En USD
		costos_operacionales_promedio_alta = costos_operacionales / total_clientes_alta
	
	elif densidad == "MEDIA":
		tecnología_media = row["Tecnología"]
		inversion_promedio_media = inversion_total / total_clientes_media		# En USD
		costos_operacionales_promedio_media = costos_operacionales / total_clientes_media	
	
	elif densidad == "BAJA":
		tecnología_baja = row["Tecnología"]
		inversion_promedio_baja = inversion_total / total_clientes_baja		# En USD
		costos_operacionales_promedio_baja = costos_operacionales / total_clientes_baja
		
	elif densidad == "MUY BAJA":
		tecnología_muy_baja = row["Tecnología"]
		inversion_promedio_muy_baja = inversion_total / total_clientes_muy_baja		# En USD
		costos_operacionales_promedio_muy_baja = costos_operacionales / total_clientes_muy_baja
		
	else:	# EXTREMADAMENTE BAJA
		tecnología_extremadamente_baja = row["Tecnología"]
		inversion_promedio_extremadamente_baja = inversion_total / total_clientes_extremadamente_baja		# En USD
		costos_operacionales_promedio_extremadamente_baja = costos_operacionales / total_clientes_extremadamente_baja

print("Costos netos promedio por tipo de densidad:")
print("----------------------------------------------------------------------")
print(f"Tecnología densidad ALTA: {tecnología_alta} | Inversión promedio [MUSD] {inversion_promedio_alta:,.2f} | Costos operacionales promedio [MUSD/año] {costos_operacionales_promedio_alta:,.2f}")
print(f"Tecnología densidad MEDIA: {tecnología_media} | Inversión promedio [MUSD] {inversion_promedio_media:,.2f} | Costos operacionales promedio [MUSD/año] {costos_operacionales_promedio_media:,.2f}")
print(f"Tecnología densidad BAJA: {tecnología_baja} | Inversión promedio [MUSD] {inversion_promedio_baja:,.2f} | Costos operacionales promedio [MUSD/año] {costos_operacionales_promedio_baja:,.2f}")
print(f"Tecnología densidad MUY BAJA: {tecnología_muy_baja} | Inversión promedio [MUSD] {inversion_promedio_muy_baja:,.2f} | Costos operacionales promedio [MUSD/año] {costos_operacionales_promedio_muy_baja:,.2f}")
print(f"Tecnología densidad EXTREMADAMENTE BAJA: {tecnología_extremadamente_baja} | Inversión promedio [MUSD] {inversion_promedio_extremadamente_baja:,.2f} | Costos operacionales promedio [MUSD/año] {costos_operacionales_promedio_extremadamente_baja:,.2f}")

Costos netos promedio por tipo de densidad:
----------------------------------------------------------------------
Tecnología densidad ALTA: G3-PLC | Inversión promedio [MUSD] 169.88 | Costos operacionales promedio [MUSD/año] 10.71
Tecnología densidad MEDIA: G3-PLC | Inversión promedio [MUSD] 186.53 | Costos operacionales promedio [MUSD/año] 10.82
Tecnología densidad BAJA: RF-Mesh | Inversión promedio [MUSD] 199.45 | Costos operacionales promedio [MUSD/año] 10.72
Tecnología densidad MUY BAJA: LoRa | Inversión promedio [MUSD] 243.05 | Costos operacionales promedio [MUSD/año] 10.88
Tecnología densidad EXTREMADAMENTE BAJA: LoRa | Inversión promedio [MUSD] 309.96 | Costos operacionales promedio [MUSD/año] 11.19


In [59]:
### Sobre el cálculo de la cantidad de brigadas necesarias para la instalación de UM
# Se declaran las productividades diarias promedio en función de la densidad y el tipo de equipo
prod_um1f_alta = 10			# UM/día en zona alta
prod_um3fd_alta = 8			# UM/día en zona alta
prod_um3fi_alta = 3			# UM/día en zona alta

prod_um1f_media = 8			# UM/día en zona media
prod_um3fd_media = 6		# UM/día en zona media
prod_um3fi_media = 3		# UM/día en zona media

prod_um1f_baja = 6			# UM/día en zona baja
prod_um3fd_baja = 4			# UM/día en zona baja
prod_um3fi_baja = 2			# UM/día en zona baja

prod_um1f_muy_baja = 4		# UM/día en zona muy baja
prod_um3fd_muy_baja = 3		# UM/día en zona muy baja	
prod_um3fi_muy_baja = 2		# UM/día en zona muy baja

prod_um1f_extremadamente_baja = 3		# UM/día en zona extremadamente baja
prod_um3fd_extremadamente_baja = 2		# UM/día en zona extremadamente baja	
prod_um3fi_extremadamente_baja = 1		# UM/día en zona extremadamente baja

objetivo_del_despliegue = 10		# Años para completar el despliegue

brigadas_zona_alta = 0
brigadas_zona_media = 0
brigadas_zona_baja = 0
brigadas_zona_muy_baja = 0
brigadas_zona_extremadamente_baja = 0

for i, row in df_implementacion.iterrows():
	densidad = row["Densidad"]
	um1f = row["Cantidad UM 1F"] / (objetivo_del_despliegue * 260)
	um3fd = row["Cantidad UM 3F Directa"] / (objetivo_del_despliegue * 260)
	um3fi = row["Cantidad UM 3F Indirecta"] / (objetivo_del_despliegue * 260)
	
	if densidad == "ALTA":
		brigadas_um1f = um1f / prod_um1f_alta
		brigadas_um3fd = um3fd / prod_um3fd_alta
		brigadas_um3fi = um3fi / prod_um3fi_alta
		brigadas_zona_alta += brigadas_um1f + brigadas_um3fd + brigadas_um3fi

	elif densidad == "MEDIA":
		brigadas_um1f = um1f / prod_um1f_media
		brigadas_um3fd = um3fd / prod_um3fd_media
		brigadas_um3fi = um3fi / prod_um3fi_media
		brigadas_zona_media += brigadas_um1f + brigadas_um3fd + brigadas_um3fi

	elif densidad == "BAJA":
		brigadas_um1f = um1f / prod_um1f_baja
		brigadas_um3fd = um3fd / prod_um3fd_baja
		brigadas_um3fi = um3fi / prod_um3fi_baja
		brigadas_zona_baja += brigadas_um1f + brigadas_um3fd + brigadas_um3fi

	elif densidad == "MUY BAJA":
		brigadas_um1f = um1f / prod_um1f_muy_baja
		brigadas_um3fd = um3fd / prod_um3fd_muy_baja
		brigadas_um3fi = um3fi / prod_um3fi_muy_baja
		brigadas_zona_muy_baja += brigadas_um1f + brigadas_um3fd + brigadas_um3fi

	else:	# EXTREMADAMENTE BAJA
		brigadas_um1f = um1f / prod_um1f_extremadamente_baja
		brigadas_um3fd = um3fd / prod_um3fd_extremadamente_baja
		brigadas_um3fi = um3fi / prod_um3fi_extremadamente_baja
		brigadas_zona_extremadamente_baja += brigadas_um1f + brigadas_um3fd + brigadas_um3fi

total_brigadas = (brigadas_zona_alta + brigadas_zona_media + brigadas_zona_baja
					+ brigadas_zona_muy_baja + brigadas_zona_extremadamente_baja)

print("Cantidad de brigadas necesarias para la instalación de UM en un plazo de {} años: {:,.2f} brigadas".format(objetivo_del_despliegue, total_brigadas))

Cantidad de brigadas necesarias para la instalación de UM en un plazo de 10 años: 523.77 brigadas


In [60]:
### Para obtener la energía promedio por unidad de cliente al año
df_energía_por_cliente = pd.DataFrame()
año = 2025

for periodo in range(len(lista_dfs_implementacion)):
	total_energia_residencial = lista_dfs_implementacion[periodo].loc[lista_dfs_implementacion[periodo]["Segmento"] == "Residencial", "Energía facturada anual [MWh/año]"].sum()
	total_clientes_residencial = lista_dfs_implementacion[periodo].loc[lista_dfs_implementacion[periodo]["Segmento"] == "Residencial", "Cantidad de clientes"].sum()
	energia_promedio_residencial = (total_energia_residencial * 1_000) / total_clientes_residencial / 12		# kWh/cliente-mes

	nueva_fila = {
		"Periodo": periodo,
		"Año": año + periodo,
		"Energía promedio residencial [kWh/cliente-mes]": energia_promedio_residencial.round(2)
	}

	df_energía_por_cliente = pd.concat([df_energía_por_cliente, pd.DataFrame([nueva_fila])], ignore_index=True)	

df_energía_por_cliente

,Periodo,Año,Energía promedio residencial [kWh/cliente-mes]
0,0,2025,187.83
1,1,2026,190.68
2,2,2027,193.57
3,3,2028,196.50
4,4,2029,199.48
5,5,2030,202.49
6,6,2031,205.56
7,7,2032,208.66
8,8,2033,211.81
9,9,2034,215.01


In [61]:
clientes_ext_baja = df_implementacion.loc[(df_implementacion["Densidad"] == "EXTREMADAMENTE BAJA") & 
                                          (df_implementacion["Segmento"] == "Residencial"), "Cantidad de clientes"].sum()
energia_ext_baja = df_implementacion.loc[(df_implementacion["Densidad"] == "EXTREMADAMENTE BAJA") & 
                                         (df_implementacion["Segmento"] == "Residencial"), "Energía facturada anual [MWh/año]"].sum()

energia_promedio_ext_baja = (energia_ext_baja * 1_000) / clientes_ext_baja / 12		# kWh/cliente-mes
energia_promedio_ext_baja

np.float64(138.07389656459083)

In [62]:
df_implementacion

Comuna           Segmento  Densidad  \
Comuna      Segmento                                                      
chimbarongo Residencial        chimbarongo        Residencial  MUY BAJA   
            No Residencial BT  chimbarongo  No Residencial BT  MUY BAJA   
            No Residencial AT  chimbarongo  No Residencial AT  MUY BAJA   
            LibreDx            chimbarongo            LibreDx  MUY BAJA   
            Red                chimbarongo                Red  MUY BAJA   
...                                    ...                ...       ...   
valdivia    Residencial           valdivia        Residencial      BAJA   
            No Residencial BT     valdivia  No Residencial BT      BAJA   
            No Residencial AT     valdivia  No Residencial AT      BAJA   
            LibreDx               valdivia            LibreDx      BAJA   
            Red                   valdivia                Red      BAJA   

                              Tipo de facturación  Superficie efectiva [km2]  \
Comuna      Segmento                                                           
chimbarongo Residencial                   Mensual                       0.00   
            No Residencial BT             Mensual                       0.00   
            No Residencial AT             Mensual                       0.00   
            LibreDx                       Mensual                       0.00   
            Red                           Mensual                     130.00   
...                                           ...                        ...   
valdivia    Residencial                 Bimensual                       0.00   
            No Residencial BT           Bimensual                       0.00   
            No Residencial AT           Bimensual                       0.00   
            LibreDx                     Bimensual                       0.00   
            Red                         Bimensual                     242.75   

                               Cantidad de clientes  UM 1F para TD  \
Comuna      Segmento                                                 
chimbarongo Residencial                       14079              0   
            No Residencial BT                   414              0   
            No Residencial AT                   299              0   
            LibreDx                               0              0   
            Red                                   0             33   
...                                             ...            ...   
valdivia    Residencial                       72814              0   
            No Residencial BT                  1554              0   
            No Residencial AT                   570              0   
            LibreDx                               0              0   
            Red                                   0            342   

                               UM 3F 1S para TD  UM 3F 2S para TD  \
Comuna      Segmento                                                
chimbarongo Residencial                       0                 0   
            No Residencial BT                 0                 0   
            No Residencial AT                 0                 0   
            LibreDx                           0                 0   
            Red                              66               222   
...                                         ...               ...   
valdivia    Residencial                       0                 0   
            No Residencial BT                 0                 0   
            No Residencial AT                 0                 0   
            LibreDx                           0                 0   
            Red                             274              1044   

                               Energía facturada anual [MWh/año]  ...  \
Comuna      Segmento                                              ...   
chimbarongo Residencial                             29186.868709

In [63]:
# Relación entre costos de inversión UM para clientes versis UM para transformadores
df_residenciales = df_implementacion[df_implementacion['Segmento'] != 'Red']
inversion_um_clientes = df_residenciales["Inversión UM 1F [USD]"].sum() + df_residenciales["Inversión UM 3F Directa [USD]"].sum() + df_residenciales["Inversión UM 3F Indirecta [USD]"].sum()
df_red = df_implementacion[df_implementacion['Segmento'] == 'Red']
inversion_um_transformadores = df_red["Inversión UM 1F [USD]"].sum() + df_red["Inversión UM 3F Directa [USD]"].sum() + df_red["Inversión UM 3F Indirecta [USD]"].sum() + df_red["Inversión DCU [USD]"].sum()

print("Inversión UM para clientes [USD]: {:,.0f}".format(inversion_um_clientes))
print("Inversión UM para transformadores [USD]: {:,.0f}".format(inversion_um_transformadores))

Inversión UM para clientes [USD]: 1,185,702,618
Inversión UM para transformadores [USD]: 220,053,945


In [64]:
# Relación entre costos de operación UM para clientes versis UM para transformadores
df_residenciales = df_implementacion[df_implementacion['Segmento'] != 'Red']
operacion_um_clientes = df_residenciales["Costos Operacionales Totales [USD/año]"].sum()
df_red = df_implementacion[df_implementacion['Segmento'] == 'Red']
operacion_um_transformadores = df_red["Costos Operacionales Totales [USD/año]"].sum()

print("Operación UM para clientes [USD]: {:,.0f}".format(operacion_um_clientes))
print("Operación UM para transformadores [USD]: {:,.0f}".format(operacion_um_transformadores))

Operación UM para clientes [USD]: 80,206,440
Operación UM para transformadores [USD]: 3,455,227


In [65]:
# Información al mes de diciembre de 2024
# -> Cantidad de clientes.
# -> Cantidad de energía.
cantidad_clientes = df_implementacion['Cantidad de clientes'].sum()
cantidad_energia = df_implementacion['Energía facturada anual [MWh/año]'].sum()

df_facturacion_clientes_residenciales = df_implementacion[df_implementacion['Segmento'] == 'Residencial']
cantidad_clientes_residenciales = df_facturacion_clientes_residenciales['Cantidad de clientes'].sum()
cantidad_energia_residenciales = df_facturacion_clientes_residenciales['Energía facturada anual [MWh/año]'].sum()

df_facturacion_clientes_no_residenciales = df_implementacion[df_implementacion['Segmento'] == 'No Residencial BT']
cantidad_clientes_no_residenciales_bt = df_facturacion_clientes_no_residenciales['Cantidad de clientes'].sum()
cantidad_energia_no_residenciales_bt = df_facturacion_clientes_no_residenciales['Energía facturada anual [MWh/año]'].sum()

df_facturacion_clientes_no_residenciales = df_implementacion[df_implementacion['Segmento'] == 'No Residencial AT']
cantidad_clientes_no_residenciales_at = df_facturacion_clientes_no_residenciales['Cantidad de clientes'].sum()
cantidad_energia_no_residenciales_at = df_facturacion_clientes_no_residenciales['Energía facturada anual [MWh/año]'].sum()

df_facturacion_clientes_no_residenciales = df_implementacion[df_implementacion['Segmento'] == 'LibreDx']
cantidad_clientes_no_residenciales_libredx = df_facturacion_clientes_no_residenciales['Cantidad de clientes'].sum()
cantidad_energia_no_residenciales_libredx = df_facturacion_clientes_no_residenciales['Energía facturada anual [MWh/año]'].sum()

cantidad_um_1f_td = df_implementacion['UM 1F para TD'].sum()
cantidad_um_3f1s_td = df_implementacion['UM 3F 1S para TD'].sum()
cantidad_um_3f2s_td = df_implementacion['UM 3F 2S para TD'].sum()

print("(A fecha de diciembre de 2024)",
      "\nClientes Distribución:", 
      "\n -> Cantidad de clientes: {:,}".format(cantidad_clientes), 
      "\n -> Energía facturada: {:,} GWh".format((cantidad_energia / 1_000).round(2)),
      "\n -> Energía promedio por cliente: {:,} kWh/cliente-mes".format((cantidad_energia * 1_000 / cantidad_clientes / 12).round(2)),
      "\nClientes Residenciales:",
      "\n -> Cantidad de clientes: {:,}".format(cantidad_clientes_residenciales),
      "\n -> Energía facturada: {:,} GWh".format((cantidad_energia_residenciales / 1_000).round(2)),
      "\n -> Energía promedio por cliente: {:,} kWh/cliente-mes".format((cantidad_energia_residenciales * 1_000 / cantidad_clientes_residenciales / 12).round(2)),
      "\nClientes No Residenciales en BT:",
      "\n -> Cantidad de clientes: {:,}".format(cantidad_clientes_no_residenciales_bt),
      "\n -> Energía facturada: {:,} GWh".format((cantidad_energia_no_residenciales_bt / 1_000).round(2)),
      "\n -> Energía promedio por cliente: {:,} kWh/cliente-mes".format((cantidad_energia_no_residenciales_bt * 1_000 / cantidad_clientes_no_residenciales_bt / 12).round(2)),
      "\nClientes No Residenciales en AT:",
	  "\n -> Cantidad de clientes: {:,}".format(cantidad_clientes_no_residenciales_at),
	  "\n -> Energía facturada: {:,} GWh".format((cantidad_energia_no_residenciales_at / 1_000).round(2)),
      "\n -> Energía promedio por cliente: {:,} kWh/cliente-mes".format((cantidad_energia_no_residenciales_at * 1_000 / cantidad_clientes_no_residenciales_at / 12).round(2)),
      "\nClientes No Residenciales LibreDx:",
      "\n -> Cantidad de clientes: {:,}".format(cantidad_clientes_no_residenciales_libredx),
      "\n -> Energía facturada: {:,} GWh".format((cantidad_energia_no_residenciales_libredx / 1_000).round(2)),
      "\n -> Energía promedio por cliente: {:,} kWh/cliente-mes".format((cantidad_energia_no_residenciales_libredx * 1_000 / cantidad_clientes_no_residenciales_libredx / 12).round(2)),
      "\n--------------------------------------------",
      "\nCantidad de UM para TD:",
	  "\n -> UM 1F para TD: {:,}".format(cantidad_um_1f_td),
	  "\n -> UM 3F 1S para TD: {:,}".format(cantidad_um_3f1s_td),
	  "\n -> UM 3F 2S para TD: {:,}".format(cantidad_um_3f2s_td),
	  "\n -> TD Totales: {:,}".format(cantidad_um_1f_td + cantidad_um_3f1s_td + cantidad_um_3f2s_td))

(A fecha de diciembre de 2024) 
Clientes Distribución: 
 -> Cantidad de clientes: 7,741,437 
 -> Energía facturada: 34,765.77 GWh 
 -> Energía promedio por cliente: 374.24 kWh/cliente-mes 
Clientes Residenciales: 
 -> Cantidad de clientes: 7,539,587 
 -> Energía facturada: 16,993.97 GWh 
 -> Energía promedio por cliente: 187.83 kWh/cliente-mes 
Clientes No Residenciales en BT: 
 -> Cantidad de clientes: 144,913 
 -> Energía facturada: 4,513.76 GWh 
 -> Energía promedio por cliente: 2,595.67 kWh/cliente-mes 
Clientes No Residenciales en AT: 
 -> Cantidad de clientes: 56,511 
 -> Energía facturada: 6,161.01 GWh 
 -> Energía promedio por cliente: 9,085.27 kWh/cliente-mes 
Clientes No Residenciales LibreDx: 
 -> Cantidad de clientes: 426 
 -> Energía facturada: 7,097.03 GWh 
 -> Energía promedio por cliente: 1,388,307.32 kWh/cliente-mes 
-------------------------------------------- 
Cantidad de UM para TD: 
 -> UM 1F para TD: 22,317 
 -> UM 3F 1S para TD: 28,932 
 -> UM 3F 2S para TD: 93,4